# 17 — Limiares de confiança por classe, sem novo treino

O experimento 16 não aprovou os checkpoints finais. Esta etapa mantém os pesos
do **candidato 10** congelados e investiga a decisão de gerar alerta.

| Propriedade | Protocolo |
| --- | --- |
| Classes ajustadas | `chair` e `table` |
| Grade de cada classe | 0,10; 0,15; 0,20; 0,25; 0,30; 0,35; 0,40 |
| Configurações | 49, somente em calibração |
| Demais classes | Fixas em 0,25, incluindo armários |
| Divisão | Grupos por SHA-256/pHash; cerca de metade para cada parte |
| Seleção | Uma configuração, congelada antes de verificar |
| Verificação | A escolha única; falhar não libera segunda tentativa |
| Treinamento | Nenhum |
| Testes e reserva | Preservados, sem inferência |

Calibração e verificação são partes das validações já usadas no desenvolvimento.
A verificação é **interna e previamente exposta**, não um novo teste independente.
Rótulos, V2, associação IoU 0,5 e pesos não serão alterados.


## 1. Executar como novo notebook no Kaggle

Ative **Internet** e habilite o Secret existente **GDRIVE_RCLONE_CONFIG_B64**.
Os caminhos já estão preenchidos. Use **Save Version → Save & Run All**.

**CPU é suficiente se os quatro caches completos do 16 estiverem disponíveis.**
GPU T4 é opcional e acelera somente uma eventual inferência de recuperação.
Se faltar cache, o notebook tenta recriá-lo com o peso original, sem treinar.
Não há formulário, correção de rótulo ou escolha manual de limiares.

O download dos ZIPs Home e da validação externa ainda é necessário para conferir
imagens, rótulos e dimensões e fazer a divisão por semelhança. Somente VAL será extraído.


In [ ]:
%pip install -q numpy pillow scipy pyyaml matplotlib


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
import sys,json,hashlib,shutil,platform,gzip,zipfile
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import PIL,scipy
from IPython.display import display,FileLink,HTML
BASE_ID='4198170ca2387a938964083e'
PASTA_LOCAL=Path('/tmp/tcc_limiares_17')/BASE_ID
PASTA_BOOT=Path('/kaggle/working/resultados_tcc')/('limiares17_preparacao_'+BASE_ID)
PASTA_LOCAL.mkdir(parents=True,exist_ok=True)
PASTA_BOOT.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS=PASTA_BOOT
DRIVE=None
REMOTO_RESULTADOS='resultados/'+PASTA_BOOT.name
AMBIENTE={'python':platform.python_version(),'numpy':np.__version__,
          'pillow':PIL.__version__,'scipy':scipy.__version__}
print('49 configurações em calibração; no máximo uma em verificação; nenhum treino.')
print('Ambiente de avaliação:',AMBIENTE)


## 2. Carregar e conferir o código

O notebook é autossuficiente. Os módulos históricos, a política V2 e o associador
estão embutidos com SHA-256. O código novo usa os nomes reais das funções históricas.
Os parâmetros abaixo fazem parte do protocolo; não é necessário editá-los.


In [ ]:
EXPECTED = {'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9', 'manifesto_pacotes': '655094dea28755a7bddc817ae66660d06fab28ccb8f9d140d3ffebb07f03a0d4', 'pacote_11': 'a3f3fb500cef83fc4890f6bbec6725bbf5e080503fd0478af3dd84d6f5ef6e93', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'codigo_11': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2'}, 'pacote_12': '1c3a6016099c2bb23aca29396286497c7973f00136c1afbb0bbc62e45aa20650', 'codigo_12': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2', 'exp10_partial.py': '86a36f93aa643853f1268dcc133bed7e7344dea269b9d1d58cf54095538fffae', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'exp12_data.py': '0a9de7e99d302a2690f18f51924f4dc80b32861c9d2149ccfe0572ec75499f30', 'exp12_training.py': '6c51c9965336b2f00bd9b282e2159c0b63f07c5c16859da78da2818bd4202fdc', 'exp12_reports.py': '120609247939189f895c01cdfd1411e4d2b416d1e293cb26851d63d04835b49a'}, 'B_e03': 'b683e1ef5fac4f8ddb4fad2f610e10dcb33bc658ea72593b8088652dbdd3c21c', 'regra_12_sha256': '04c6287c2c4adb8c43baf28c10acbaa1a599198b905d48808a8f5f67f39a4909', 'pacote_15': '266c0b4a0b8e7ac7fa940a009e0a009c22ab1cd17f403a4917d4ab3abe514487', 'pacote_16': 'cd9acf4eba29bb28dee20465ec2503b72cc65302f3d7c8377dd442455a4d16fa', 'contexto_16': '44a814e0a728085209af51a3da632ef7568722115204a93e6a223ef49fde399b', 'pasta_16': 'resultados/controle16_44a814e0a728085209af51a3', 'pasta_10': 'resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010', 'pasta_11': 'resultados/diagnostico11_20261016_024603_683411', 'associador': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d'}
MODULOS_EMBUTIDOS = {'conf14_data.py': '"""Entradas autenticadas do diagnóstico 14; extrai somente Home VAL."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom PIL import Image\nimport diag11_core as core\n\nCOHORTS = (\'home_val_historica\', \'openimages_val_nova\')\nBOOLS = {\'True\':True, \'False\':False}\n\ndef rows(raw):\n    return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\n\ndef parse_events(raw):\n    result = rows(raw)\n    for r in result:\n        r[\'classe_id\'] = int(r[\'classe_id\'])\n        for field in [\'bbox_ref\',\'bbox_pred\']:\n            r[field] = json.loads(r[field]) if r[field] else None\n        for field in [\'ha_alerta_ref\',\'ha_alerta_pred\']:\n            r[field] = BOOLS[r[field]]\n        r[\'confianca\'] = float(r[\'confianca\']) if r[\'confianca\'] else None\n    return result\n\ndef gt_key(r):\n    return (r[\'id\'],int(r[\'classe_id\']),tuple(round(float(v),6) for v in r[\'bbox_ref\']))\n\ndef gt_map(events):\n    result = {}\n    for r in events:\n        if r[\'bbox_ref\'] is None: continue\n        key = gt_key(r)\n        if key in result: raise ValueError(\'Objeto GT repetido nos registros.\')\n        result[key] = r\n    return result\n\ndef load_source12(path, expected, output=None):\n    if core.sha256_file(path)!=expected[\'pacote_12\']:\n        raise RuntimeError(\'O pacote 12 mudou em relação ao recebido.\')\n    bundle = {\'evaluations\':{}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):\n            raise RuntimeError(\'Pacote 12 com membros repetidos.\')\n        for name in z.namelist(): core.safe_member(name)\n        if z.testzip() is not None: raise RuntimeError(\'Falha de CRC no pacote 12.\')\n        cfg = json.loads(z.read(\'configuracao_experimento_12.json\'))\n        summary = json.loads(z.read(\'resumo_experimento_12.json\'))\n        if not summary[\'concluido\'] or summary[\'reserva_futura_inferida\'] or summary[\'test_home_inferido\'] or summary[\'test_oi20k_inferido\']:\n            raise RuntimeError(\'Execução 12 incompatível com o protocolo.\')\n        if cfg[\'expected\'][\'candidato\']!=expected[\'candidato\'] or cfg[\'expected\'][\'politica\']!=expected[\'politica\'] or summary[\'dataset_sha256\']!=expected[\'dataset_novo\']:\n            raise RuntimeError(\'Referências ou política diferentes do experimento 12.\')\n        if cfg[\'codigo_embutido_sha256\']!=expected[\'codigo_12\']:\n            raise RuntimeError(\'Código do experimento 12 diferente do recebido.\')\n        for name, digest in expected[\'codigo_12\'].items():\n            if hashlib.sha256(z.read(\'codigo/\'+name)).hexdigest()!=digest:\n                raise RuntimeError(\'Código alterado no pacote 12: \'+name)\n        rule_raw = z.read(\'regra_selecao_congelada.json\')\n        if hashlib.sha256(rule_raw).hexdigest()!=cfg[\'regra_sha256\']:\n            raise RuntimeError(\'A regra original do experimento 12 foi alterada.\')\n        selection = json.loads(z.read(\'selecao_checkpoints.json\'))\n        checkpoint = next(e for e in selection[\'assessed\'] if e[\'id\']==\'B_home_replay_e03\')\n        if checkpoint[\'weight_sha256\']!=expected[\'B_e03\']:\n            raise RuntimeError(\'Checkpoint B_e03 diferente do esperado.\')\n        for model,prefix in [(\'baseline\',\'referencias/baseline\'),(\'candidato_10\',\'referencias/candidato_10\'),(\'B_e03\',\'avaliacoes/B_home_replay_e03\')]:\n            bundle[\'evaluations\'][model] = {}\n            for cohort in COHORTS:\n                base=prefix+\'/\'+cohort+\'/\'\n                members={name:z.read(base+name) for name in [\'metricas.json\',\'alertas_por_imagem.csv\',\'avaliacao_objetos.csv\']}\n                metrics=json.loads(members[\'metricas.json\'])\n                images=rows(members[\'alertas_por_imagem.csv\'])\n                for r in images:\n                    for key in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']:\n                        r[key]=int(r[key])\n                if len({r[\'id\'] for r in images})!=len(images) or len(images)!=metrics[\'imagens\']:\n                    raise RuntimeError(\'IDs ou quantidades inconsistentes nos registros 12.\')\n                for field, n in [(\'tp_alerta\',metrics[\'alertas_por_objeto\'][\'tp\']),(\'fn_alerta\',metrics[\'alertas_por_objeto\'][\'fn\']),(\'fp_alerta\',metrics[\'alertas_por_objeto\'][\'fp\']),(\'alta_ref\',metrics[\'alta_ref\']),(\'alta_sem_alerta\',metrics[\'alta_sem_alerta\'])]:\n                    if sum(r[field] for r in images)!=n: raise RuntimeError(\'Contagens inconsistentes no pacote 12.\')\n                events=parse_events(members[\'avaliacao_objetos.csv\'])\n                gt_map(events)\n                bundle[\'evaluations\'][model][cohort]={\'summary\':metrics,\'images\':images,\'events\':events}\n                if output is not None:\n                    dest=Path(output)/\'referencias_12\'/model/cohort\n                    dest.mkdir(parents=True,exist_ok=True)\n                    for name, raw in members.items(): (dest/name).write_bytes(raw)\n        for cohort in COHORTS:\n            target=gt_map(bundle[\'evaluations\'][\'B_e03\'][cohort][\'events\'])\n            for model in [\'baseline\',\'candidato_10\']:\n                source=gt_map(bundle[\'evaluations\'][model][cohort][\'events\'])\n                if source.keys()!=target.keys(): raise RuntimeError(\'Modelos do 12 usam referências GT diferentes.\')\n                if any(source[k][\'ha_alerta_ref\']!=target[k][\'ha_alerta_ref\'] or source[k][\'prioridade_ref\']!=target[k][\'prioridade_ref\'] for k in target):\n                    raise RuntimeError(\'Referências de prioridade diferentes entre modelos.\')\n        bundle.update(config=cfg,summary=summary,rule=json.loads(rule_raw),\n                      audit=json.loads(z.read(\'auditoria_dados.json\')))\n    return bundle\n\ndef extract_home_val(path, destination, expected_zip, expected_test, expected_policy):\n    if core.sha256_file(path)!=expected_zip: raise RuntimeError(\'ZIP Home alterado.\')\n    destination=Path(destination)\n    records=[];frozen=[]\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())): raise RuntimeError(\'Membros repetidos no Home.\')\n        for name in z.namelist():core.safe_member(name)\n        manifest=json.loads(z.read(\'verificacao_preparacao.json\'))\n        if (manifest[\'test\'][\'sha256\']!=expected_test or manifest[\'test\'][\'imagens\']!=403 or\n            {int(k):v for k,v in manifest[\'classes\'].items()}!=dict(enumerate(core.NAMES))):\n            raise RuntimeError(\'Manifesto Home incompatível.\')\n        root=manifest[\'dataset_dir\']\n        items={r[\'arquivo\']:r for r in manifest[\'arquivos\']}\n        if len(items)!=len(manifest[\'arquivos\']): raise RuntimeError(\'Manifesto Home repetido.\')\n        train_hash=set();val_hash=set();all_hash=set();counts=Counter()\n        for name,item in items.items():\n            is_image=name.startswith(root+\'/images/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}\n            if not is_image:continue\n            all_hash.add(item[\'sha256\'])\n            if name.startswith(root+\'/images/train/raw_train/\'):\n                counts[\'train\']+=1;train_hash.add(item[\'sha256\'])\n            elif name.startswith(root+\'/images/val/raw_train/\'):\n                counts[\'val\']+=1;val_hash.add(item[\'sha256\'])\n        if counts!={\'train\':1824,\'val\':456} or train_hash&val_hash:\n            raise RuntimeError(\'A divisão atual de Home mudou ou tem bytes repetidos entre TRAIN e VAL.\')\n        for name in z.namelist():\n            if not(name.startswith(\'codigo/\') and name.endswith(\'.py\')):continue\n            raw=z.read(name)\n            if name in items and hashlib.sha256(raw).hexdigest()!=items[name][\'sha256\']:\n                raise RuntimeError(\'Código Home alterado.\')\n            out=destination/name;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n            frozen.append((out,hashlib.sha256(raw).hexdigest()))\n        for name,item in sorted(items.items()):\n            if not(name.startswith(root+\'/images/val/raw_train/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}):continue\n            label=name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            if label not in items:raise RuntimeError(\'Label Home ausente.\')\n            paths={}\n            for field,member in [(\'image\',name),(\'label\',label)]:\n                raw=z.read(member)\n                if hashlib.sha256(raw).hexdigest()!=items[member][\'sha256\']:raise RuntimeError(\'Home alterado: \'+member)\n                out=destination/member;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n                paths[field]=out;frozen.append((out,items[member][\'sha256\']))\n            with Image.open(paths[\'image\']) as im:\n                width,height=im.size;im.verify()\n            records.append({\'id\':\'home:\'+name,\'name\':name.split(\'/raw_train/\')[-1],**paths,\n                \'role\':\'home_val\',\'coorte\':\'home_val_historica\',\'width\':width,\'height\':height,\n                \'known_mask\':core.ALL_KNOWN,\'sha256\':item[\'sha256\'],\'label_sha256\':items[label][\'sha256\'],\n                \'gt\':core.parse_yolo(paths[\'label\'].read_bytes(),width,height,source=label)})\n    if len(records)!=456 or core.sha256_file(destination/\'codigo/src/politica_alertas.py\')!=expected_policy:\n        raise RuntimeError(\'Quantidade Home VAL ou política V2 incompatível.\')\n    audit={\'home_train_manifesto\':1824,\'home_val_extraida\':456,\'home_train_extraida\':False,\n           \'home_test_extraido\':False,\'test_home_manifesto\':403,\'intersecao_bytes_train_val\':0,\n           \'historico_original_baseline_nao_comprovado_pela_separacao_atual\':True}\n    return records,all_hash,frozen,audit\n\ndef validate_cohorts(cohorts,bundle,reserved_ids):\n    if set(cohorts)!=set(COHORTS) or {k:len(v) for k,v in cohorts.items()}!={\'home_val_historica\':456,\'openimages_val_nova\':1000}:\n        raise ValueError(\'Coortes diferentes das duas validações congeladas.\')\n    for cohort,records in cohorts.items():\n        ids={r[\'id\'] for r in records}\n        if len(ids)!=len(records) or ids&set(reserved_ids):raise ValueError(\'ID repetido ou reservado na avaliação.\')\n        expected_role=\'home_val\' if cohort==\'home_val_historica\' else \'val_nova\'\n        if any(r[\'role\']!=expected_role for r in records):raise ValueError(\'Papel incompatível com a coorte de validação.\')\n        old=bundle[\'evaluations\'][\'B_e03\'][cohort]\n        if ids!={r[\'id\'] for r in old[\'images\']}:raise ValueError(\'IDs diferentes dos avaliados no 12.\')\n        actual={}\n        for r in records:\n            for g in r[\'gt\']:\n                if r[\'known_mask\']&(1<<g[\'classe_id\']):\n                    k=(r[\'id\'],g[\'classe_id\'],tuple(round(float(x),6) for x in g[\'bbox\']))\n                    if k in actual:raise ValueError(\'GT repetido na entrada atual.\')\n                    actual[k]=True\n        if actual.keys()!=gt_map(old[\'events\']).keys():raise ValueError(\'Caixas avaliáveis diferentes das referências originais.\')\n    return True\n', 'conf14_evaluation.py': '"""Varredura de confiança: uma inferência por imagem, política V2 fixa."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nfrom exp12_protocol import judge\n\nTHRESHOLDS=(.05,.10,.15,.20,.25,.30,.40)\nPREDICT_ARGS={\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.70,\'max_det\':300,\'half\':False,\'batch\':1}\nVERSION=\'14-confianca-automatica-1.0\'\n\ndef point_name(conf):return \'conf_\'+f\'{conf:.2f}\'.replace(\'.\',\'\')\n\n\ndef cache_request(records,weight_sha,environment):\n    if not records or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Cache exige IDs únicos.\')\n    if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in records):raise ValueError(\'Cache somente para validação.\')\n    signature=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':VERSION,\'peso_sha256\':weight_sha,\'parametros\':PREDICT_ARGS,\n            \'ambiente\':environment,\'coorte\':records[0][\'coorte\'],\'imagens\':len(records),\n            \'entradas_sha256\':hashlib.sha256(core.canonical(signature).encode()).hexdigest()}\n\ndef read_cache(path,records):\n    expected={r[\'id\']:r for r in records};result={}\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            row=json.loads(line);iid=row[\'id\']\n            if iid not in expected or iid in result:raise ValueError(\'Cache com ID desconhecido ou repetido.\')\n            r=expected[iid]\n            if row[\'image_sha256\']!=r[\'sha256\'] or row[\'label_sha256\']!=r[\'label_sha256\']:\n                raise ValueError(\'Cache pertence a outros bytes de imagem/label.\')\n            validate_predictions(row[\'predicoes\'],r);result[iid]=row[\'predicoes\']\n    if result.keys()!=expected.keys():raise ValueError(\'Cache incompleto.\')\n    return result\n\n\ndef event(record,kind,g,p,overlap,pred_low,conf):\n    row=core.record_event(record,\'B_e03\',kind,g,p,overlap,pred_low)\n    row[\'limiar_confianca\']=conf\n    row[\'melhor_confianca_mesma_classe\']=None\n    if kind==\'objeto_nao_detectado\':\n        flags=[v for v in row[\'indicios\'].split(\'|\') if v and v not in {\'deteccao_abaixo_da_confianca_025\',\'sobreposicao_com_previsao_de_outra_classe\',\'omissao_sem_indicio_automatico\'}]\n        matrix=core.iou_matrix([g],pred_low)[0]\n        same=[i for i,p0 in enumerate(pred_low) if p0[\'classe_id\']==g[\'classe_id\']]\n        if same:\n            best=max(same,key=lambda i:matrix[i]);row[\'melhor_confianca_mesma_classe\']=pred_low[best][\'confianca\']\n            if matrix[best]>=.5 and pred_low[best][\'confianca\']<conf:flags.append(\'deteccao_abaixo_do_limiar_avaliado\')\n        if any(matrix[i]>=.5 and p0[\'classe_id\']!=g[\'classe_id\'] and p0[\'confianca\']>=conf for i,p0 in enumerate(pred_low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\']=\'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record,pred_low,conf,policy,matcher):\n    if record[\'role\'] not in {\'home_val\',\'val_nova\'} or conf not in THRESHOLDS:\n        raise ValueError(\'Papel ou limiar fora do protocolo.\')\n    known=int(record[\'known_mask\'])\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in pred_low if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in pred_low if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[event(record,\'correspondencia\',gt[gi],pred[pi],float(iou),low,conf) for gi,pi,iou in pairs]\n    events += [event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    messages=[policy.gerar_mensagem(p,\'medio\') for p in all_pred if core.alert_on(p)]\n    if any(not isinstance(s,str) or not s.strip() for s in messages):raise RuntimeError(\'Alerta sem mensagem.\')\n    im={\'coorte\':record[\'coorte\'],\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':\'B_e03\',\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return im,events\n\ndef summarise(images,events,conf):\n    if not images:raise ValueError(\'Validação vazia.\')\n    total=lambda k:sum(r[k] for r in images)\n    complete=[r for r in images if r[\'cobertura_completa_alertas\']]\n    summary={\'modelo\':\'B_e03\',\'coorte\':images[0][\'coorte\'],\'imagens\':len(images),\'conf_alertas\':conf,\n        \'alertas_por_objeto\':core.alert_metrics(total(\'tp_alerta\'),total(\'fn_alerta\'),total(\'fp_alerta\')),\n        \'alta_ref\':total(\'alta_ref\'),\'alta_sem_alerta\':total(\'alta_sem_alerta\'),\n        \'taxa_alta_sem_alerta\':core.ratio(total(\'alta_sem_alerta\'),total(\'alta_ref\')),\n        \'detector_no_limiar\':core.alert_metrics(total(\'tp_detector\'),total(\'fn_detector\'),total(\'fp_detector\')),\n        \'gt_ignorados_sem_cobertura\':total(\'gt_ignorados\'),\'predicoes_ignoradas_conf\':total(\'predicoes_ignoradas_conf\'),\n        \'imagens_com_cobertura_completa_alertas\':len(complete),\n        \'alertas_por_imagem_cobertura_completa\':core.alert_metrics(\n            sum(r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete),\n            sum(r[\'ha_alerta_ref\'] and not r[\'ha_alerta_pred\'] for r in complete),\n            sum(not r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete)),\n        \'referencia\':\'Concordância com política V2 nas caixas; não valida risco físico.\',\n        \'nao_e_openimages_challenge\':True,\'ap_recalculada\':False}\n    classes=[]\n    for cid,name in enumerate(core.NAMES):\n        part=[e for e in events if e[\'classe_id\']==cid];n=sum(e[\'bbox_ref\']!=\'\' for e in part)\n        classes.append({\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':n,\'suporte_reduzido\':n<30,\'conf_alertas\':conf,\n            **core.alert_metrics(sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part),\n                                 sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in part),\n                                 sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part)),\n            \'alta_ref\':sum(e[\'prioridade_ref\']==\'alto\' for e in part),\n            \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in part)})\n    return summary,classes\n\ndef evaluate_cache(records,cache,conf,policy,matcher,output=None):\n    if cache.keys()!={r[\'id\'] for r in records}:raise ValueError(\'Cache não corresponde aos IDs da validação.\')\n    images=[];events=[]\n    for r in records:\n        im,ev=analyse(r,cache[r[\'id\']],conf,policy,matcher);images.append(im);events.extend(ev)\n    summary,classes=summarise(images,events,conf)\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'metricas.json\',summary)\n        core.save_csv(output/\'metricas_por_classe.csv\',classes)\n        core.save_csv(output/\'alertas_por_imagem.csv\',images)\n        core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    return {\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n\ndef parse_current_events(events):\n    result=[]\n    for r in events:\n        v=dict(r)\n        for key in [\'bbox_ref\',\'bbox_pred\']:v[key]=json.loads(v[key]) if v[key] else None\n        result.append(v)\n    return result\n\ndef check_reproduction(result,original,cohort):\n    old={r[\'id\']:r for r in original[\'images\']};new={r[\'id\']:r for r in result[\'images\']}\n    if old.keys()!=new.keys():raise ValueError(\'Reprodução exige os mesmos IDs.\')\n    rows=[]\n    fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']\n    for iid in sorted(old):\n        for key in fields:\n            key_new=\'predicoes_ignoradas_conf\' if key==\'predicoes_ignoradas_conf_025\' else key\n            a,b=old[iid][key],new[iid][key_new]\n            rows.append({\'coorte\':cohort,\'id\':iid,\'metrica\':key,\'notebook12\':a,\'notebook14\':b,\'igual\':a==b})\n    old_gt=data.gt_map(original[\'events\']);new_gt=data.gt_map(parse_current_events(result[\'events\']))\n    if old_gt.keys()!=new_gt.keys():raise ValueError(\'Referências GT mudaram na reprodução.\')\n    object_rows=[]\n    for key,a in old_gt.items():\n        b=new_gt[key]\n        fields_gt=[\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']\n        diffs=[field for field in fields_gt if a[field]!=b[field]]\n        object_rows.append({\'coorte\':cohort,\'id\':key[0],\'classe\':a[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                            \'campos_diferentes\':\'|\'.join(diffs),\'igual\':not diffs})\n    return {\'coorte\':cohort,\'checagens_imagem\':len(rows),\'divergencias_imagem\':sum(not r[\'igual\'] for r in rows),\n            \'objetos_GT\':len(object_rows),\'divergencias_objetos\':sum(not r[\'igual\'] for r in object_rows),\n            \'reproduziu\':all(r[\'igual\'] for r in rows+object_rows)},rows,object_rows\n\ndef select_threshold(results,rule,reproduction_ok):\n    assessed=[]\n    for conf in THRESHOLDS:\n        home=results[\'home_val_historica\'][conf][\'summary\'];ext=results[\'openimages_val_nova\'][conf][\'summary\']\n        supported=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n                      for s in [home,ext] for k in [\'precision\',\'recall\',\'f1\'])\n        if supported:\n            checks={**judge(home,ext,rule),\'metricas_com_suporte\':True}\n        else:\n            # Precisão sem nenhum alerta é indefinida; não vira zero ou aprovação.\n            checks={k:None for k in [\'home_precision_superior\',\'home_recall_preservado\',\'home_altas_preservadas\',\n                                    \'externa_precision_preservada\',\'externa_recall_preservado\',\'externa_altas_preservadas\']}\n            checks.update(elegivel=False,retencao_externa=False,metricas_com_suporte=False)\n        assessed.append({\'conf\':conf,\'home\':home,\'external\':ext,\'checks\':checks,\n                         \'comparavel_historico\':reproduction_ok,\'elegivel\':checks[\'elegivel\'] and reproduction_ok})\n    def ranking(r):\n        h=r[\'home\'][\'alertas_por_objeto\'];e=r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],r[\'home\'][\'alta_sem_alerta\'],-e[\'f1\'],abs(r[\'conf\']-.25),-r[\'conf\'])\n    good=[r for r in assessed if r[\'elegivel\']]\n    chosen=min(good,key=ranking) if good else None\n    status=\'limiar_candidato_em_validacao\' if chosen else \'nenhum_limiar_elegivel\'\n    if not reproduction_ok:status=\'reproducao_divergente_sem_sugestao_de_limiar\'\n    return {\'status\':status,\'limiar_sugerido\':chosen[\'conf\'] if chosen else None,\n            \'comparabilidade_confirmada\':bool(reproduction_ok),\'assessed\':assessed,\n            \'limiar_galeria\':chosen[\'conf\'] if chosen else .10,\n            \'criterio_galeria_sem_elegivel\':\'0,10 fixado antes da inferência; somente diagnóstico.\',\n            \'modelo_em_uso_continua_baseline\':True,\'substituicao_automatica\':False}\n\ndef effects(results,bundle,cohort,cache):\n    old=data.gt_map(bundle[\'evaluations\'][\'candidato_10\'][cohort][\'events\'])\n    anchor=data.gt_map(parse_current_events(results[cohort][.25][\'events\']))\n    lost={k for k in anchor if old[k][\'ha_alerta_ref\'] and old[k][\'ha_alerta_pred\'] and not anchor[k][\'ha_alerta_pred\']}\n    changed=[];losses=[];priority=[]\n    for key in sorted(lost):\n        a,b=old[key],anchor[key]\n        if b[\'evento\']==\'correspondencia\':\n            priority.append({\'coorte\':cohort,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_candidato10\':a[\'prioridade_pred\'],\'prioridade_B_025\':b[\'prioridade_pred\'],\n                \'confianca_candidato10\':a[\'confianca\'],\'confianca_B_025\':b[\'confianca\'],\n                \'observacao\':\'Objeto ainda associado; alerta pode mudar pela geometria/prioridade.\'})\n    for conf in THRESHOLDS:\n        current=data.gt_map(parse_current_events(results[cohort][conf][\'events\']))\n        for key,b in current.items():\n            a=anchor[key]\n            if a[\'ha_alerta_ref\'] and a[\'ha_alerta_pred\']!=b[\'ha_alerta_pred\']:\n                changed.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                    \'alerta_B_025\':a[\'ha_alerta_pred\'],\'alerta_no_limiar\':b[\'ha_alerta_pred\'],\n                    \'prioridade_ref\':b[\'prioridade_ref\'],\'confianca_associada\':b[\'confianca\']})\n            if key not in lost:continue\n            same=[p for p in cache[key[0]] if p[\'classe_id\']==key[1]]\n            ious=core.iou_matrix([{\'bbox\':list(key[2])}],same)[0]\n            best=max(range(len(same)),key=lambda i:ious[i]) if same else None\n            losses.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_ref\':b[\'prioridade_ref\'],\'recuperado\':b[\'ha_alerta_pred\'],\n                \'evento_B_025\':a[\'evento\'],\'evento_no_limiar\':b[\'evento\'],\n                \'melhor_iou_cache_mesma_classe\':float(ious[best]) if best is not None else None,\n                \'confianca_dessa_proposta\':same[best][\'confianca\'] if best is not None else None,\n                \'limite\':\'Melhor IoU não garante associação nem alerta; coluna recuperado usa avaliação completa.\'})\n    return changed,losses,priority\n\n\ndef validate_predictions(pred,record):\n    """Valida estrutura; preserva coordenadas numéricas como no avaliador 11/12.\n\n    Área zero e limites da imagem são diagnósticos, não filtros do cache.\n    Nenhuma proposta é removida, recortada ou corrigida aqui.\n    """\n    totals={\'predicoes\':len(pred),\'area_zero\':0,\'fora_da_imagem\':0,\n            \'area_zero_na_grade\':0,\'fora_da_imagem_na_grade\':0,\'amostras\':[]}\n    for index,p in enumerate(pred):\n        reason=None\n        if not isinstance(p,dict):\n            raise ValueError(f"Previsão não é um dicionário: {record[\'id\']}; índice={index}; valor={p!r}")\n        cid=p.get(\'classe_id\');box=p.get(\'bbox\');score=p.get(\'confianca\')\n        if type(cid)!=int or not 0<=cid<len(core.NAMES):reason=\'classe_id inválida\'\n        elif p.get(\'classe\')!=core.NAMES[cid]:reason=\'nome da classe incompatível\'\n        elif type(score) not in (int,float) or not math.isfinite(score) or not .001-1e-8<=score<=1:\n            reason=\'confiança não finita ou fora da faixa de captura\'\n        elif not isinstance(box,(list,tuple)) or len(box)!=4:reason=\'bbox deve ter quatro coordenadas\'\n        elif not all(type(v) in (int,float) and math.isfinite(v) for v in box):reason=\'coordenada não numérica ou não finita\'\n        elif box[2]<box[0] or box[3]<box[1]:reason=\'coordenadas xyxy invertidas\'\n        if reason:\n            raise ValueError(f"Previsão incompatível com o cache: {record[\'id\']}; motivo={reason}; "\n                f"índice={index}; classe_id={cid!r}; confiança={score!r}; bbox={box!r}; "\n                f"dimensões=(w={record[\'width\']}, h={record[\'height\']})")\n        zero=box[2]==box[0] or box[3]==box[1]\n        outside=(min(box)<-.01 or box[2]>record[\'width\']+.01 or box[3]>record[\'height\']+.01)\n        totals[\'area_zero\']+=int(zero);totals[\'fora_da_imagem\']+=int(outside)\n        totals[\'area_zero_na_grade\']+=int(zero and score>=min(THRESHOLDS))\n        totals[\'fora_da_imagem_na_grade\']+=int(outside and score>=min(THRESHOLDS))\n        if (zero or outside) and len(totals[\'amostras\'])<5:\n            totals[\'amostras\'].append({\'indice\':index,\'classe\':p[\'classe\'],\'confianca\':score,\n                \'bbox\':list(box),\'area_zero\':zero,\'fora_da_imagem\':outside,\n                \'width\':record[\'width\'],\'height\':record[\'height\']})\n    return totals\n\ndef save_geometry_diagnostics(directory,items):\n    totals={key:sum(item[\'diagnostico\'][key] for item in items) for key in\n            [\'predicoes\',\'area_zero\',\'fora_da_imagem\',\'area_zero_na_grade\',\'fora_da_imagem_na_grade\']}\n    unusual=[item for item in items if item[\'diagnostico\'][\'area_zero\'] or item[\'diagnostico\'][\'fora_da_imagem\']]\n    report={\'correcao\':\'14-cache-validacao-1.0.1\',\'imagens\':len(items),\n        \'imagens_com_geometria_incomum\':len(unusual),\'totais\':totals,\n        \'propostas_removidas\':0,\'coordenadas_alteradas\':False,\n        \'observacao\':\'Geometria incomum preservada no cache e no protocolo; não constitui objeto aprovado. Reprodução em 0,25 continua obrigatória.\',\n        \'casos\':unusual}\n    core.save_json(Path(directory)/\'diagnostico_geometria_previsoes.json\',report)\n    return {k:v for k,v in report.items() if k!=\'casos\'}\n\ndef obtain_cache(model,records,directory,weight_sha,environment,device,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    path=directory/\'predicoes_conf_0p001.jsonl.gz\';meta_path=directory/\'cache_manifesto.json\'\n    request=cache_request(records,weight_sha,environment)\n    if path.is_file() and meta_path.is_file():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'solicitacao\']!=request or meta[\'cache_sha256\']!=core.sha256_file(path):\n            raise RuntimeError(\'Cache existente não corresponde aos pesos/entradas/ambiente. Não será reutilizado.\')\n        cached=read_cache(path,records)\n        diagnostics=[{\'id\':r[\'id\'],\'diagnostico\':validate_predictions(cached[r[\'id\']],r)} for r in records]\n        geometry=save_geometry_diagnostics(directory,diagnostics)\n        print(\'Cache conferido e reutilizado:\',records[0][\'coorte\'],flush=True)\n        return path,{**meta,\'geometria_resumo\':geometry,\'reutilizado_nesta_execucao\':True}\n    predict=predict_fn or core.model_predictions\n    tmp=path.with_name(path.name+\'.tmp\')\n    saturated=0;diagnostics=[]\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        for i,r in enumerate(records,1):\n            if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel proibido na inferência.\')\n            pred=predict(model,r,device)\n            try:diagnostic=validate_predictions(pred,r)\n            except ValueError as exc:\n                core.save_json(directory/\'previsao_rejeitada.json\',{\'id\':r[\'id\'],\'mensagem\':str(exc),\n                    \'width\':r[\'width\'],\'height\':r[\'height\'],\'propostas\':pred})\n                raise\n            diagnostics.append({\'id\':r[\'id\'],\'diagnostico\':diagnostic})\n            if len(pred)==PREDICT_ARGS[\'max_det\']:saturated+=1\n            f.write(json.dumps({\'id\':r[\'id\'],\'imagem\':r[\'name\'],\'image_sha256\':r[\'sha256\'],\n                                \'label_sha256\':r[\'label_sha256\'],\'predicoes\':pred},ensure_ascii=False)+\'\\n\')\n            if i%100==0 or i==len(records):print(f"Checkpoint / {r[\'coorte\']}: {i}/{len(records)}",flush=True)\n    tmp.replace(path)\n    geometry=save_geometry_diagnostics(directory,diagnostics)\n    meta={\'solicitacao\':request,\'cache_sha256\':core.sha256_file(path),\'concluido\':True,\n          \'validacao_previsoes\':\'14-cache-validacao-1.0.1\',\'geometria_resumo\':geometry,\n          \'imagens_atingiram_max_det\':saturated,\'limite\':\'Propostas pós-NMS em confiança 0,001; max_det=300, sem acesso aos logits brutos.\'}\n    core.save_json(meta_path,meta)\n    read_cache(path,records)\n    print(\'Diagnóstico geométrico do cache:\',geometry[\'totais\'],flush=True)\n    return path,{**meta,\'reutilizado_nesta_execucao\':False}\n', 'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'diag11_data.py': '"""Entradas, auditoria documental e nova validação, sem consulta a previsões."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport time\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nfrom diag11_core import (ALL_KNOWN, ALERT_MASK, NAMES, canonical, parse_yolo,\n                         safe_member, save_csv, save_json, sha256_file)\n\nSOURCE_URLS = {\n    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",\n    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",\n    "caixas": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",\n    "rotulos": "https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels-boxable.csv",\n    "metadata": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",\n}\n\n\ndef download(url, path, retries=3):\n    """Download atômico, com recibo local e validação de tamanho quando informado."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".download.json")\n    if path.is_file() and receipt.is_file():\n        prior = json.loads(receipt.read_text())\n        if prior.get("url") == url and prior.get("sha256") == sha256_file(path):\n            return path\n    last = None\n    for attempt in range(retries):\n        tmp = path.with_name(path.name + ".tmp")\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "TCC-Diagnostico-11/1.0"})\n            with urllib.request.urlopen(req, timeout=45) as response, tmp.open("wb") as out:\n                expected = response.headers.get("Content-Length")\n                for block in iter(lambda: response.read(2**20), b""):\n                    out.write(block)\n            if expected and tmp.stat().st_size != int(expected):\n                raise IOError("Tamanho recebido difere do Content-Length.")\n            if not tmp.stat().st_size:\n                raise IOError("Arquivo recebido vazio.")\n            tmp.replace(path)\n            save_json(receipt, {"url": url, "sha256": sha256_file(path),\n                                "bytes": path.stat().st_size})\n            return path\n        except Exception as exc:\n            last = type(exc).__name__ + ": " + str(exc)[:180]\n            tmp.unlink(missing_ok=True)\n            if attempt + 1 < retries:\n                time.sleep(min(2**attempt, 4))\n    raise IOError(f"Download não concluído: {url}; {last}")\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as f:\n        yield from csv.DictReader(f)\n\n\ndef load_history(csv_path, json_path, expected_hash):\n    split = json.loads(Path(json_path).read_text())\n    stated = split.pop("divisao_sha256")\n    computed = hashlib.sha256(canonical(split).encode()).hexdigest()\n    if stated != expected_hash or computed != stated:\n        raise RuntimeError("Divisão histórica não corresponde à execução 10.")\n    rows = list(csv_rows(csv_path))\n    if len(rows) != 100000 or len({r["id"] for r in rows}) != 100000:\n        raise RuntimeError("O manifesto histórico precisa de 100.000 IDs distintos.")\n    if Counter(r["split"] for r in rows) != {"train": 80000, "test": 20000}:\n        raise RuntimeError("Divisão histórica diferente de 80/20.")\n    for label, key in [("train", "ids_treino"), ("test", "ids_teste")]:\n        if {r["id"] for r in rows if r["split"] == label} != set(split[key]):\n            raise RuntimeError("CSV e JSON históricos divergem.")\n    hashes = {r["imagem_sha256"] for r in rows}\n    if len(hashes) != len(rows):\n        raise RuntimeError("O histórico contém imagens repetidas em bytes.")\n    return {"ids": {r["id"] for r in rows}, "hashes": hashes, "rows": rows,\n            "split_hash": stated}\n\n\ndef phash(path):\n    with Image.open(path) as image:\n        small = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),\n                           dtype=float)\n    low = dctn(small, norm="ortho")[:8, :8]\n    bits = low > np.median(low.flatten()[1:])\n    bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), "big")\n\n\nclass NearIndex:\n    """pHash de 64 bits, distância <=4; sinal de semelhança, não prova semântica."""\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, name):\n        for band in range(8):\n            self.tables[band][(value >> (8*band)) & 255].append((value, name))\n\n    def find(self, value, max_distance=4):\n        checked = set()\n        for band in range(8):\n            for prior, name in self.tables[band].get((value >> (8*band)) & 255, []):\n                if (prior, name) in checked:\n                    continue\n                checked.add((prior, name))\n                if (prior ^ value).bit_count() <= max_distance:\n                    return name\n        return None\n\n\ndef extract_home(zip_path, destination, expected_zip, expected_test, expected_policy):\n    if sha256_file(zip_path) != expected_zip:\n        raise RuntimeError("ZIP Home diferente da entrada preservada.")\n    destination = Path(destination)\n    records, hashes, near = [], set(), NearIndex()\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Há membros repetidos no ZIP Home.")\n        manifest = json.loads(z.read("verificacao_preparacao.json"))\n        if (manifest["test"]["sha256"] != expected_test or\n                manifest["test"]["imagens"] != 403 or\n                {int(k): v for k, v in manifest["classes"].items()} != dict(enumerate(NAMES))):\n            raise RuntimeError("Manifesto Home incompatível.")\n        root = manifest["dataset_dir"]\n        items = {r["arquivo"]: r for r in manifest["arquivos"]}\n        if len(items) != len(manifest["arquivos"]):\n            raise RuntimeError("Manifesto Home com membros repetidos.")\n        for name in z.namelist():\n            safe_member(name)\n            if name.startswith("codigo/") and name.endswith(".py"):\n                raw = z.read(name)\n                if name in items and hashlib.sha256(raw).hexdigest() != items[name]["sha256"]:\n                    raise RuntimeError("Código Home diferente do manifesto.")\n                out = destination / name\n                out.parent.mkdir(parents=True, exist_ok=True)\n                out.write_bytes(raw)\n        for name, item in sorted(items.items()):\n            is_image = (name.startswith(f"{root}/images/") and\n                        Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})\n            if is_image:\n                hashes.add(item["sha256"])\n            role = None\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                role = "home_train"\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                role = "home_val"\n            if not is_image or role is None:\n                continue\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise RuntimeError(f"Imagem Home alterada: {name}")\n            image = destination / name\n            image.parent.mkdir(parents=True, exist_ok=True)\n            image.write_bytes(raw)\n            label_name = name.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"\n            if label_name not in items:\n                raise RuntimeError("Label Home ausente do manifesto.")\n            label_raw = z.read(label_name)\n            if hashlib.sha256(label_raw).hexdigest() != items[label_name]["sha256"]:\n                raise RuntimeError("Label Home alterado.")\n            with Image.open(image) as im:\n                width, height = im.size\n                im.verify()\n            gt = parse_yolo(label_raw, width, height, source=label_name)\n            record = {"id": "home:" + name, "name": name.split("/raw_train/")[-1],\n                      "image": image, "role": role, "coorte": "home_val_historica",\n                      "width": width, "height": height, "known_mask": ALL_KNOWN,\n                      "sha256": item["sha256"], "label_sha256": items[label_name]["sha256"],\n                      "gt": gt}\n            record["phash"] = phash(image)\n            near.add(record["phash"], name)\n            records.append(record)\n    counts = Counter(r["role"] for r in records)\n    if counts != {"home_train": 1824, "home_val": 456}:\n        raise RuntimeError(f"Home inesperado: {dict(counts)}")\n    train_hash = {r["sha256"] for r in records if r["role"] == "home_train"}\n    val_hash = {r["sha256"] for r in records if r["role"] == "home_val"}\n    if train_hash & val_hash:\n        raise RuntimeError("Home TRAIN e VAL atuais compartilham imagens em bytes.")\n    policy_file = destination / "codigo/src/politica_alertas.py"\n    if not policy_file.is_file() or sha256_file(policy_file) != expected_policy:\n        raise RuntimeError("Política V2 diferente da versão preservada.")\n    audit = {"home_train_atual": 1824, "home_val_atual": 456,\n             "test_home_preservado": 403, "test_extraido": False,\n             "intersecao_bytes_home_train_val_atual": 0,\n             "divergencia_documental_1822_1824": "Não reconciliada; o pacote atual tem 1824.",\n             "separacao_atual_nao_comprova_historico_baseline": True}\n    return records, hashes, near, audit\n\n\ndef audit_baseline_metadata(checkpoint, expected_hash, home_records, evidence=None):\n    if sha256_file(checkpoint) != expected_hash:\n        raise RuntimeError("Peso baseline diferente da referência.")\n    # Só o checkpoint autenticado pelo SHA esperado é desserializado.\n    import torch\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    args = obj.get("train_args", {})\n    fields = ["data", "epochs", "batch", "imgsz", "seed", "optimizer", "lr0",\n              "project", "name", "fraction"]\n    audit = {"modelo_sha256": expected_hash, "data_checkpoint": str(obj.get("date")),\n             "train_args_preservados": {k: args[k] for k in fields if k in args},\n             "estado_proveniencia_original": "nao_determinado",\n             "home_val_usada_no_treino_original": None,\n             "motivo": "Caminhos e argumentos não contêm a lista de imagens vistas no treino original."}\n    del obj\n    if evidence is not None:\n        required = {"modelo_sha256", "train_image_sha256", "val_image_sha256"}\n        if not required <= evidence.keys() or evidence["modelo_sha256"] != expected_hash:\n            audit["evidencia_original"] = "incompativel_ou_sem_vinculo_ao_peso"\n        else:\n            train = set(evidence["train_image_sha256"])\n            val = set(evidence["val_image_sha256"])\n            valid_sha = lambda x: isinstance(x, str) and len(x) == 64 and all(c in "0123456789abcdef" for c in x)\n            if not train or any(not valid_sha(h) for h in train | val):\n                raise ValueError("Manifesto original precisa de hashes SHA-256 válidos.")\n            current = {r["sha256"] for r in home_records if r["role"] == "home_val"}\n            overlap = train & current\n            complete = evidence.get("manifesto_treino_completo") is True\n            audit.update(estado_proveniencia_original="manifesto_documental_analisado",\n                         evidencia_original="documento_vinculado_ao_sha_do_modelo",\n                         home_val_usada_no_treino_original=True if overlap else (False if complete else None),\n                         manifesto_declarado_completo=complete,\n                         imagens_home_val_no_manifesto_train_original=len(overlap),\n                         imagens_home_val_no_manifesto_val_original=len(current & val),\n                         motivo="Conclusão limitada à completude e correção do manifesto apresentado.")\n    return audit\n\n\ndef stable_order(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef balanced_ids(records, n, seed=42, per_class=45):\n    """Diversidade por classe e preenchimento determinístico, sem usar previsões."""\n    if len(records) < n:\n        raise ValueError(f"Há somente {len(records)} imagens elegíveis; exigidas {n}.")\n    groups = {c: sorted((iid for iid, r in records.items() if\n                         r["positive_mask"] & (1 << c)), key=lambda x: stable_order(x, seed))\n              for c in range(12)}\n    order = sorted(groups, key=lambda c: (len(groups[c]), c))\n    chosen, counts = set(), Counter()\n    # Diversificar exemplos negativos quando as seis classes têm ausência verificada.\n    negatives = sorted((iid for iid, r in records.items() if\n                        r["known_mask"] & ALERT_MASK == ALERT_MASK and\n                        not r["positive_mask"] & ALERT_MASK), key=lambda x: stable_order(x, seed))\n    for iid in negatives[:min(len(negatives), n//10)]:\n        chosen.add(iid)\n        counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n    for target in range(1, per_class+1):\n        for cid in order:\n            if len(chosen) >= n:\n                break\n            if counts[cid] >= target:\n                continue\n            for iid in groups[cid]:\n                if iid not in chosen:\n                    chosen.add(iid)\n                    counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n                    break\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return sorted(chosen, key=lambda x: stable_order(x, seed))\n\n\ndef split_pool(records, n_reserve, seed=42):\n    """Estratos pela classe positiva mais rara; a reserva não entra na inferência."""\n    frequencies = Counter(c for r in records.values() for c in range(12)\n                          if r["positive_mask"] & (1 << c))\n    groups = defaultdict(list)\n    for iid, record in records.items():\n        classes = record["classes_positivas_avaliaveis"]\n        primary = min(classes, key=lambda c: (frequencies[c], c)) if classes else -1\n        groups[primary].append(iid)\n    total = len(records)\n    if not 0 < n_reserve < total:\n        raise ValueError("A reserva precisa ser menor que o conjunto novo.")\n    quotas = {c: len(ids)*n_reserve//total for c, ids in groups.items()}\n    order = sorted(groups, key=lambda c: (-(len(groups[c])*n_reserve % total), c))\n    for cid in order[:n_reserve-sum(quotas.values())]:\n        quotas[cid] += 1\n    selected = set()\n    for cid, ids in groups.items():\n        selected.update(sorted(ids, key=lambda x: stable_order(x, seed+1))[:quotas[cid]])\n    if len(selected) != n_reserve:\n        raise RuntimeError("A divisão não produziu a reserva planejada.")\n    return {iid: ("reserva_futura" if iid in selected else "val_nova") for iid in records}\n\n\ndef filter_metadata(records, metadata_path, prior_ids):\n    metadata, exclusions = {}, Counter()\n    for row in csv_rows(metadata_path):\n        iid = row["ImageID"]\n        if iid not in records:\n            continue\n        if row["Subset"] != "validation":\n            raise ValueError("A metadata contém uma origem diferente de validation.")\n        if iid in prior_ids:\n            exclusions["id_presente_nos_100k"] += 1\n            continue\n        try:\n            rotation = float(row["Rotation"])\n        except (TypeError, ValueError):\n            rotation = float("nan")\n        # Evita suposições sobre transformação das caixas e orientação do JPEG.\n        if rotation != 0:\n            exclusions["rotacao_nao_zero_ou_desconhecida"] += 1\n            continue\n        metadata[iid] = {k: row.get(k, "") for k in\n                         ["ImageID", "OriginalLandingURL", "License", "Author", "Rotation"]}\n    result = {iid: {**r, "creditos": metadata[iid]} for iid, r in records.items()\n              if iid in metadata}\n    return result, dict(exclusions)\n\n\ndef download_images(candidates, ids, directory, workers=8):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def one(iid):\n        record = dict(candidates[iid])\n        path = directory / f"oi_val_{iid}.jpg"\n        try:\n            download(f"https://open-images-dataset.s3.amazonaws.com/validation/{iid}.jpg", path)\n            with Image.open(path) as image:\n                image.verify()\n            with Image.open(path) as image:\n                width, height = image.size\n                orientation = image.getexif().get(274, 1)\n                if width < 32 or height < 32 or orientation != 1:\n                    raise ValueError("JPEG pequeno ou com orientação EXIF não trivial.")\n                image.load()\n            tiny = {g["classe_id"] for g in record["gt_normalizado"] if\n                    (g["bbox_norm"][2]-g["bbox_norm"][0])*width < 1 or\n                    (g["bbox_norm"][3]-g["bbox_norm"][1])*height < 1}\n            # A avaliação da ausência fica desconhecida para a classe afetada.\n            record["restricoes_cobertura"] = dict(record["restricoes_cobertura"])\n            for cid in tiny:\n                for key in ["known_mask", "positive_mask", "negative_mask"]:\n                    record[key] &= ~(1 << cid)\n                record["restricoes_cobertura"][str(cid)] = ["caixa_menor_que_um_pixel"]\n            record["classes_positivas_avaliaveis"] = [c for c in range(12)\n                                                      if record["positive_mask"] & (1 << c)]\n            if not (record["positive_mask"] or record["known_mask"] & ALERT_MASK == ALERT_MASK):\n                raise ValueError("Nenhuma cobertura útil após conferir as dimensões reais.")\n            record.update(image=path, width=width, height=height, sha256=sha256_file(path),\n                          phash=phash(path), name=path.name)\n            return record, None\n        except Exception as exc:\n            return None, {"id": iid, "motivo": type(exc).__name__, "detalhe": str(exc)[:240]}\n\n    valid, failures = {}, []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for done, (record, failure) in enumerate(executor.map(one, ids), 1):\n            if record:\n                valid[record["id"]] = record\n            else:\n                failures.append(failure)\n            if done % 100 == 0 or done == len(ids):\n                print(f"Novas imagens: {done}/{len(ids)}; válidas: {len(valid)}", flush=True)\n    return valid, failures\n\n\ndef deduplicate(records, prior_hashes, home_near, seed=42):\n    accepted, excluded, seen_hash, near = {}, [], set(), NearIndex()\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        record = records[iid]\n        reason, similar = None, None\n        if record["sha256"] in prior_hashes:\n            reason = "bytes_presentes_nos_dados_anteriores"\n        elif record["sha256"] in seen_hash:\n            reason = "bytes_repetidos_no_conjunto_novo"\n        elif (similar := home_near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_com_home_train_ou_val"\n        elif (similar := near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_dentro_do_conjunto_novo"\n        if reason:\n            excluded.append({"id": iid, "motivo": reason, "semelhante_a": similar})\n            continue\n        accepted[iid] = record\n        seen_hash.add(record["sha256"])\n        near.add(record["phash"], iid)\n    return accepted, excluded\n\n\ndef freeze_dataset(records, roles, directory, source_hashes, prior_split_hash, seed=42):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest, runtime = [], []\n    for iid in sorted(roles):\n        r, role = records[iid], roles[iid]\n        boxes = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            boxes.append({**g, "bbox": [x1*r["width"], y1*r["height"],\n                                       x2*r["width"], y2*r["height"]]})\n        coverage = {k: r[k] for k in ["known_mask", "positive_mask", "negative_mask",\n                                      "restricoes_cobertura"]}\n        label = directory / "labels" / role / f"oi_val_{iid}.txt"\n        coverage_file = directory / "coverage" / role / f"oi_val_{iid}.json"\n        label.parent.mkdir(parents=True, exist_ok=True)\n        lines = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            lines.append(f"{g[\'classe_id\']} {(x1+x2)/2:.10f} {(y1+y2)/2:.10f} {(x2-x1):.10f} {(y2-y1):.10f}")\n        label.write_text("\\n".join(lines)+("\\n" if lines else ""), encoding="utf-8")\n        save_json(coverage_file, coverage)\n        image_rel = f"images/{role}/oi_val_{iid}.jpg"\n        item = {"id": iid, "role": role, "image": image_rel,\n                "label": label.relative_to(directory).as_posix(),\n                "coverage": coverage_file.relative_to(directory).as_posix(),\n                "sha256": r["sha256"], "label_sha256": sha256_file(label),\n                "coverage_sha256": sha256_file(coverage_file),\n                "width": r["width"], "height": r["height"], "known_mask": r["known_mask"],\n                "positive_mask": r["positive_mask"], "negative_mask": r["negative_mask"],\n                "phash": f"{r[\'phash\']:016x}", "creditos": r["creditos"],\n                "gt_normalizado": r["gt_normalizado"]}\n        manifest.append(item)\n        runtime.append({**r, "role": role, "coorte": "openimages_val_nova",\n                        "gt": boxes, "label": label, "coverage": coverage_file})\n    metadata = {"versao": "11-validacao-independente-v1", "seed": seed, "nomes": NAMES,\n                "fontes_metadata_sha256": source_hashes, "origem_split": "validation",\n                "divisao_100k_anterior_sha256": prior_split_hash,\n                "contagens": dict(Counter(roles.values())),\n                "criterio": "Diversidade de classes, hash determinístico e pHash; sem previsões",\n                "cobertura": "Positivo explícito na raiz ou negativo na raiz/ancestral; classes problemáticas excluídas",\n                "independencia_limitada_ao_historico_conhecido": True,\n                "pretreinamentos_publicos_nao_auditados": True,\n                "reserva_futura_avaliada": False, "arquivos": manifest}\n    dataset_hash = hashlib.sha256(canonical(metadata).encode()).hexdigest()\n    metadata["dataset_sha256"] = dataset_hash\n    save_json(directory / "manifesto_validacao_nova.json", metadata)\n    save_csv(directory / "divisao.csv", [{k: r[k] for k in\n                                        ["id", "role", "sha256", "known_mask", "positive_mask"]}\n                                       for r in manifest])\n    archive = directory.parent / f"validacao_independente_11_{dataset_hash[:12]}.zip"\n    # Os JPEGs permanecem sem recompressão.\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:\n        z.write(directory / "manifesto_validacao_nova.json", "manifesto_validacao_nova.json")\n        z.write(directory / "divisao.csv", "divisao.csv")\n        for r, item in zip(sorted(runtime, key=lambda r: r["id"]), manifest):\n            z.write(r["image"], item["image"])\n            z.write(r["label"], item["label"])\n            z.write(r["coverage"], item["coverage"])\n    return runtime, metadata, archive\n\n\ndef support_rows(records):\n    rows = []\n    for role in sorted({r["role"] for r in records}):\n        group = [r for r in records if r["role"] == role]\n        for cid, name in enumerate(NAMES):\n            gt = [g for r in group for g in r["gt"] if\n                  g["classe_id"] == cid and r["known_mask"] & (1 << cid)]\n            rows.append({"conjunto": role, "classe_id": cid, "classe": name,\n                         "imagens_cobertura_conhecida": sum(bool(r["known_mask"] & (1 << cid)) for r in group),\n                         "imagens_positivas_avaliaveis": sum(bool(r["positive_mask"] & (1 << cid)) for r in group),\n                         "caixas_avaliaveis": len(gt), "suporte_reduzido": len(gt) < 30})\n    return rows\n', 'exp12_data.py': '"""Entradas congeladas e replay somente do TRAIN histórico."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport yaml\n\nimport diag11_core as core\nimport diag11_data as d11\nfrom exp12_protocol import diverse_subset\n\n\ndef check_digest(raw, expected, label):\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise RuntimeError("Conteúdo alterado: " + label)\n\n\ndef materialise_home_labels(zip_path, home):\n    with zipfile.ZipFile(zip_path) as z:\n        for record in home:\n            member = record["id"].removeprefix("home:")\n            label_name = member.replace("/images/", "/labels/").rsplit(".", 1)[0]+".txt"\n            core.safe_member(label_name)\n            raw = z.read(label_name)\n            check_digest(raw, record["label_sha256"], label_name)\n            label = Path(str(record["image"]).replace("/images/", "/labels/")).with_suffix(".txt")\n            label.parent.mkdir(parents=True, exist_ok=True)\n            label.write_bytes(raw)\n            record["label"] = label\n\n\ndef read_package11(path, expected_sha, expected_models, expected_dataset):\n    if core.sha256_file(path) != expected_sha:\n        raise RuntimeError("O pacote do diagnóstico 11 mudou.")\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Pacote 11 com membros repetidos.")\n        for name in z.namelist():\n            core.safe_member(name)\n        summary = json.loads(z.read("resumo_diagnostico_11.json"))\n        config = json.loads(z.read("configuracao_diagnostico.json"))\n        dataset_location = json.loads(z.read("localizacao_dataset.json"))\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if (config["modelos_sha256"] != expected_models or\n                config["dataset_sha256"] != expected_dataset or\n                dataset_location["dataset_sha256"] != expected_dataset or\n                manifest["dataset_sha256"] != expected_dataset or\n                summary["reserva_futura_inferida"] is not False):\n            raise RuntimeError("Identidades ou reserva do diagnóstico 11 incompatíveis.")\n    return summary, config, dataset_location, manifest\n\n\ndef load_frozen_validation(zip_path, destination, expected_archive_sha, expected_dataset_sha,\n                           expected_manifest):\n    if core.sha256_file(zip_path) != expected_archive_sha:\n        raise RuntimeError("ZIP da validação independente diferente do congelado no 11.")\n    destination = Path(destination)\n    val, reserved_ids, frozen_files = [], set(), []\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Dataset novo com membros repetidos.")\n        for member in z.namelist():\n            core.safe_member(member)\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if manifest != expected_manifest:\n            raise RuntimeError("O manifesto do pacote difere do diagnóstico 11.")\n        raw_manifest = dict(manifest)\n        claimed = raw_manifest.pop("dataset_sha256")\n        if hashlib.sha256(core.canonical(raw_manifest).encode()).hexdigest() != claimed or claimed != expected_dataset_sha:\n            raise RuntimeError("Hash canônico da validação nova não confere.")\n        if manifest["nomes"] != core.NAMES or manifest["origem_split"] != "validation":\n            raise RuntimeError("Classes ou origem da validação alteradas.")\n        items = manifest["arquivos"]\n        if (Counter(r["role"] for r in items) != {"val_nova": 1000, "reserva_futura": 300}\n                or len({r["id"] for r in items}) != 1300\n                or len({r["sha256"] for r in items}) != 1300):\n            raise RuntimeError("Identidades ou quantidades do conjunto independente incompatíveis.")\n        for item in items:\n            role = item["role"]\n            for field, prefix in [("image", "images"), ("label", "labels"), ("coverage", "coverage")]:\n                if not item[field].startswith(prefix+"/"+role+"/"):\n                    raise RuntimeError("Caminho do dataset não corresponde ao papel declarado.")\n                if item[field] not in z.namelist():\n                    raise RuntimeError("Arquivo declarado ausente no ZIP.")\n            if role == "reserva_futura":\n                reserved_ids.add(item["id"])\n                continue  # Nenhum JPEG, label ou cobertura da reserva é extraído.\n            paths = {}\n            for field, hash_field in [("image", "sha256"), ("label", "label_sha256"),\n                                      ("coverage", "coverage_sha256")]:\n                raw = z.read(item[field])\n                check_digest(raw, item[hash_field], item[field])\n                output = destination/item[field]\n                output.parent.mkdir(parents=True, exist_ok=True)\n                output.write_bytes(raw)\n                paths[field] = output\n                frozen_files.append((output, item[hash_field]))\n            coverage = json.loads(paths["coverage"].read_text())\n            if any(coverage[k] != item[k] for k in ["known_mask", "positive_mask", "negative_mask"]):\n                raise RuntimeError("Cobertura da validação foi alterada.")\n            with Image.open(paths["image"]) as image:\n                if image.size != (item["width"], item["height"]):\n                    raise RuntimeError("Dimensões diferentes do manifesto.")\n                image.verify()\n            parsed = core.parse_yolo(paths["label"].read_bytes(), item["width"], item["height"])\n            if len(parsed) != len(item["gt_normalizado"]):\n                raise RuntimeError("Quantidade de caixas difere do manifesto.")\n            gt = []\n            for original, label_obj in zip(item["gt_normalizado"], parsed):\n                a, b, c, d = original["bbox_norm"]\n                box = [a*item["width"], b*item["height"], c*item["width"], d*item["height"]]\n                if (original["classe_id"] != label_obj["classe_id"] or\n                        not np.allclose(box, label_obj["bbox"], rtol=0, atol=.001)):\n                    raise RuntimeError("Caixa do manifesto diverge do label autenticado.")\n                gt.append({**original, "bbox": box})\n            val.append({**item, **paths, "gt": gt, "role": "val_nova",\n                        "coorte": "openimages_val_nova", "name": paths["image"].name})\n    return val, reserved_ids, frozen_files\n\n\ndef load_replay(package_items, selected_history, local_dir, fetch_package, expected_cfg,\n                forbidden_ids, forbidden_hashes, protected_near, validator, final_size=2000, seed=42):\n    """Lê só os membros escolhidos dos cinco lotes existentes, com SHA integral."""\n    local_dir = Path(local_dir)\n    local_dir.mkdir(parents=True, exist_ok=True)\n    wanted = {r["id"]: r for r in selected_history}\n    protected_ids, protected_hashes = set(forbidden_ids), set(forbidden_hashes)\n    records, exclusions = {}, []\n    frozen, seen, package_receipts = [], set(), []\n    for item in package_items:\n        pkg = local_dir/item["arquivo"]\n        fetch_package(item, pkg)\n        if pkg.stat().st_size != item["zip_bytes"] or core.sha256_file(pkg) != item["zip_sha256"]:\n            raise RuntimeError("Pacote de replay incompleto ou alterado.")\n        with zipfile.ZipFile(pkg) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError("Lote de replay com membros repetidos.")\n            for name in z.namelist():\n                core.safe_member(name)\n            manifest = json.loads(z.read("manifesto_lote.json"))\n            if manifest["config_sha256"] != expected_cfg or manifest["origem_split"] != "train":\n                raise RuntimeError("Lote de replay não pertence ao TRAIN autenticado.")\n            for r in manifest["aceitas"]:\n                if r["id"] not in wanted:\n                    continue\n                if r["id"] in seen:\n                    raise RuntimeError("ID de replay repetido entre lotes.")\n                seen.add(r["id"])\n                prior = wanted[r["id"]]\n                if (prior["split"] != "train" or prior["lote"] != item["arquivo"] or\n                        prior["imagem_sha256"] != r["sha256"] or\n                        int(prior["known_mask"]) != r["known_mask"]):\n                    raise RuntimeError("Replay diverge da divisão 80/20 histórica.")\n                if r["id"] in protected_ids or r["sha256"] in protected_hashes:\n                    raise RuntimeError("Replay coincide com dado protegido.")\n                raw_image, raw_label, raw_cov = (z.read(r[k]) for k in ["imagem", "label", "cobertura"])\n                for raw, key in [(raw_image, "sha256"), (raw_label, "label_sha256"), (raw_cov, "cobertura_sha256")]:\n                    check_digest(raw, r[key], r["id"]+"/"+key)\n                classes = validator(raw_label, source=item["arquivo"]+":"+r["label"])\n                cov = json.loads(raw_cov)\n                known_classes = (set(cov["classes_positivas_verificadas_exaustivas"]) |\n                                 set(cov["classes_negativas_verificadas"]))\n                if (not all(isinstance(cid, int) and 0 <= cid < 12 for cid in known_classes)\n                        or sum(1 << cid for cid in known_classes) != r["known_mask"]\n                        or cov["known_mask"] != r["known_mask"]\n                        or set(classes) & set(cov["classes_negativas_verificadas"])):\n                    raise RuntimeError("Máscara ou anotação parcial de replay incompatível.")\n                image = local_dir/"images/train"/f"oi_{r[\'id\']}.jpg"\n                label = local_dir/"labels/train"/f"oi_{r[\'id\']}.txt"\n                coverage_file = local_dir/"coverage/train"/f"oi_{r[\'id\']}.json"\n                for path, raw in [(image, raw_image), (label, raw_label), (coverage_file, raw_cov)]:\n                    path.parent.mkdir(parents=True, exist_ok=True)\n                    path.write_bytes(raw)\n                with Image.open(image) as im:\n                    width, height = im.size\n                    im.verify()\n                if (width, height) != (r["width"], r["height"]):\n                    raise RuntimeError("Dimensões de replay incompatíveis.")\n                near = protected_near.find(d11.phash(image))\n                if near:\n                    exclusions.append({"id": r["id"], "motivo": "semelhanca_phash_com_validacao_ou_reserva", "semelhante_a": near})\n                    continue\n                gt = core.parse_yolo(raw_label, width, height)\n                records[r["id"]] = {\n                    **prior, "oi_id": r["id"], "id": r["id"], "source": "openimages",\n                    "image": image, "label": label, "coverage": coverage_file,\n                    "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "label_sha256": r["label_sha256"], "coverage_sha256": r["cobertura_sha256"],\n                    "width": width, "height": height, "classes_com_caixa": classes,\n                    "gt": gt, "name": image.name, "role": "train", "coorte": "replay_train",\n                }\n        package_receipts.append(item)\n        pkg.unlink()\n    if seen != set(wanted):\n        raise RuntimeError("Alguns IDs planejados não foram encontrados nos cinco lotes.")\n    if len(records) < final_size:\n        raise RuntimeError(f"Replay com somente {len(records)} elegíveis; exigidos {final_size}.")\n    chosen = diverse_subset(list(records.values()), final_size, seed)\n    output = {Path(r["image"]).resolve().as_posix(): r for r in chosen}\n    for record in output.values():\n        frozen += [(record["image"], record["sha256"]), (record["label"], record["label_sha256"]),\n                   (record["coverage"], record["coverage_sha256"])]\n    return output, exclusions, frozen, package_receipts\n\n\ndef make_training_inputs(home, replay, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    home_train = {Path(r["image"]).resolve().as_posix():\n                  {**r, "source": "home", "known_mask": core.ALL_KNOWN}\n                  for r in home if r["role"] == "home_train"}\n    val = [Path(r["image"]).resolve().as_posix() for r in home if r["role"] == "home_val"]\n    if len(home_train) != 1824 or len(val) != 456:\n        raise RuntimeError("Home diferente da configuração congelada.")\n    val_file = destination/"home_val.txt"\n    val_file.write_text("\\n".join(sorted(val))+"\\n", encoding="utf-8")\n    arms = {}\n    for name, records in [("A_home", home_train), ("B_home_replay", {**home_train, **replay})]:\n        path = destination/(name+".txt")\n        path.write_text("\\n".join(sorted(records))+"\\n", encoding="utf-8")\n        data_file = destination/(name+".yaml")\n        data_file.write_text(yaml.safe_dump({"path": str(destination.resolve()), "train": str(path.resolve()),\n                                           "val": str(val_file.resolve()), "names": dict(enumerate(core.NAMES))},\n                                          sort_keys=False, allow_unicode=True), encoding="utf-8")\n        arms[name] = {"records": records, "yaml": data_file}\n    return arms\n\n\ndef restore_result(package_path, cohort, model, cache_dir):\n    cache_dir = Path(cache_dir)/cohort/model\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(package_path) as z:\n        prefix = cohort+"/"+model+"/"\n        summary = json.loads(z.read(prefix+"metricas.json"))\n        def rows(name):\n            return list(csv.DictReader(io.StringIO(z.read(prefix+name).decode("utf-8-sig"))))\n        images, events, classes = rows("alertas_por_imagem.csv"), rows("avaliacao_objetos.csv"), rows("metricas_por_classe.csv")\n        for row in images:\n            for k in ["tp_alerta", "fn_alerta", "fp_alerta", "alta_ref", "alta_sem_alerta", "tp_detector",\n                      "fn_detector", "fp_detector", "gt_avaliaveis", "gt_ignorados", "predicoes_ignoradas_conf_025"]:\n                row[k] = int(row[k])\n            for k in ["cobertura_completa_alertas", "ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        for row in events:\n            row["classe_id"] = int(row["classe_id"])\n            for k in ["ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        cache = cache_dir/"predicoes_conf_0001.jsonl.gz"\n        cache.write_bytes(z.read(prefix+"predicoes_conf_0001.jsonl.gz"))\n    return {"summary": summary, "images": images, "events": events, "classes": classes, "cache_file": cache}\n\n\ndef verify_frozen(files):\n    for path, sha in files:\n        if core.sha256_file(path) != sha:\n            raise RuntimeError("Arquivo protegido alterado: "+str(path))\n', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'lim17_evaluation.py': '"""Correspondências refeitas para cada limiar, usando a V2 original."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\nimport diag11_core as core\nimport conf14_evaluation as historical\nimport lim17_protocol as protocol\n\ndef _guard(record):\n    expected = {\'home_val_historica\': \'home_val\', \'openimages_val_nova\': \'val_nova\'}\n    if record[\'coorte\'] not in expected or record[\'role\'] != expected[record[\'coorte\']]:\n        raise ValueError(\'Somente as duas validações autorizadas podem ser avaliadas.\')\n\ndef _event(record, name, kind, gt, pred, overlap, low, thresholds):\n    row = core.record_event(record, name, kind, gt, pred, overlap, low)\n    cid = int(row[\'classe_id\'])\n    threshold = thresholds[cid]\n    row[\'limiar_confianca_classe\'] = threshold\n    row[\'melhor_confianca_mesma_classe\'] = None\n    if kind == \'objeto_nao_detectado\':\n        excluded = {\'deteccao_abaixo_da_confianca_025\', \'sobreposicao_com_previsao_de_outra_classe\',\n                    \'omissao_sem_indicio_automatico\'}\n        flags = [v for v in row[\'indicios\'].split(\'|\') if v and v not in excluded]\n        overlaps = core.iou_matrix([gt], low)[0]\n        same = [i for i, p in enumerate(low) if p[\'classe_id\'] == cid]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            row[\'melhor_confianca_mesma_classe\'] = low[best][\'confianca\']\n            if overlaps[best] >= .5 and low[best][\'confianca\'] < threshold:\n                flags.append(\'deteccao_abaixo_do_limiar_da_classe\')\n        if any(overlaps[i] >= .5 and p[\'classe_id\'] != cid\n               and p[\'confianca\'] >= thresholds[p[\'classe_id\']] for i, p in enumerate(low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\'] = \'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record, predictions, thresholds, policy, matcher, name):\n    _guard(record)\n    p = protocol.validate_profile(thresholds)\n    known = int(record[\'known_mask\'])\n    gt = core.with_policy([g for g in record[\'gt\'] if known & (1 << g[\'classe_id\'])],\n                          record[\'width\'], record[\'height\'], policy)\n    low = [v for v in predictions if known & (1 << v[\'classe_id\'])]\n    all_pred = core.with_policy([v for v in predictions if v[\'confianca\'] >= p[v[\'classe_id\']]],\n                                record[\'width\'], record[\'height\'], policy)\n    pred = [v for v in all_pred if known & (1 << v[\'classe_id\'])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = [_event(record, name, \'correspondencia\', gt[gi], pred[pi], float(iou), low, p)\n              for gi, pi, iou in pairs]\n    events += [_event(record, name, \'objeto_nao_detectado\', gt[gi], None, None, low, p) for gi in missing]\n    events += [_event(record, name, \'previsao_nao_associada\', None, pred[pi], None, low, p) for pi in extras]\n    messages = [policy.gerar_mensagem(v, \'medio\') for v in all_pred if core.alert_on(v)]\n    if any(not isinstance(s, str) or not s.strip() for s in messages):\n        raise RuntimeError(\'Alerta sem mensagem.\')\n    image = {\'coorte\': record[\'coorte\'], \'imagem\': record[\'name\'], \'id\': record[\'id\'], \'modelo\': name,\n             \'partition\': record.get(\'partition\', \'completa_descritiva\'),\n             \'perfil_limiares\': protocol.profile_id(p),\n             \'limiar_cadeira\': p[2], \'limiar_mesa\': p[3],\n             \'tp_alerta\': sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n             \'fn_alerta\': sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n             \'fp_alerta\': sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n             \'alta_ref\': sum(g[\'prioridade\'] == \'alto\' for g in gt),\n             \'alta_sem_alerta\': sum(e[\'prioridade_ref\'] == \'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n             \'tp_detector\': len(pairs), \'fn_detector\': len(missing), \'fp_detector\': len(extras),\n             \'gt_avaliaveis\': len(gt), \'gt_ignorados\': len(record[\'gt\']) - len(gt),\n             \'predicoes_ignoradas_conf\': len(all_pred) - len(pred),\n             \'cobertura_completa_alertas\': known & core.ALERT_MASK == core.ALERT_MASK,\n             \'ha_alerta_ref\': any(core.alert_on(g) for g in gt),\n             \'ha_alerta_pred\': any(core.alert_on(v) for v in pred), \'mensagem_alerta\': \'\\n\'.join(messages)}\n    return image, events\n\ndef evaluate(records, cache, thresholds, policy, matcher, name, output=None):\n    p = protocol.validate_profile(thresholds)\n    if not records or any(r[\'id\'] not in cache for r in records):\n        raise ValueError(\'Cache incompleto ou avaliação vazia.\')\n    images, events = [], []\n    for record in records:\n        im, ev = analyse(record, cache[record[\'id\']], p, policy, matcher, name)\n        images.append(im)\n        events.extend(ev)\n    summary, classes = historical.summarise(images, events, .25)\n    summary.update({\'modelo\': name, \'conf_alertas\': .25 if p == protocol.profile() else None,\n                    \'conf_alertas_por_classe\': p, \'perfil_limiares\': protocol.profile_id(p),\n                    \'partition\': images[0][\'partition\'], \'ap_recalculada\': False})\n    for row in classes:\n        row[\'conf_alertas\'] = p[row[\'classe_id\']]\n    result = {\'summary\': summary, \'classes\': classes, \'images\': images, \'events\': events}\n    if output:\n        output = Path(output)\n        core.save_json(output / \'metricas.json\', summary)\n        core.save_csv(output / \'metricas_por_classe.csv\', classes)\n        core.save_csv(output / \'alertas_por_imagem.csv\', images)\n        core.save_csv(output / \'avaliacao_objetos.csv\', events)\n    return result\n\ndef _counts(gt, pred, matcher):\n    pairs, missing, extras = matcher(gt, pred)\n    states = [(gt[i], pred[j]) for i, j, _ in pairs]\n    states += [(gt[i], None) for i in missing]\n    states += [(None, pred[j]) for j in extras]\n    return {\'tp_alerta\': sum(core.alert_on(g) and core.alert_on(p) for g, p in states),\n            \'fn_alerta\': sum(core.alert_on(g) and not core.alert_on(p) for g, p in states),\n            \'fp_alerta\': sum(not core.alert_on(g) and core.alert_on(p) for g, p in states),\n            \'alta_ref\': sum(g[\'prioridade\'] == \'alto\' for g in gt),\n            \'alta_sem_alerta\': sum(bool(g and g[\'prioridade\'] == \'alto\') and not core.alert_on(p) for g, p in states),\n            \'tp_detector\': len(pairs), \'fn_detector\': len(missing), \'fp_detector\': len(extras),\n            \'gt_avaliaveis\': len(gt), \'gt_ignorados\': 0}\n\ndef calibration_frontier(records, cache, policy, matcher, progress=None):\n    """Pré-calcula por classe: o associador original não cruza classes."""\n    if not records or any(r.get(\'partition\') != \'calibracao\' for r in records):\n        raise ValueError(\'A grade só pode pré-calcular a partição de calibração.\')\n    book = []\n    for index, r in enumerate(records, 1):\n        _guard(r)\n        gt = core.with_policy([g for g in r[\'gt\'] if r[\'known_mask\'] & (1 << g[\'classe_id\'])], r[\'width\'], r[\'height\'], policy)\n        # A política só recebe propostas que podem entrar na grade; geometrias do cache não são modificadas.\n        eligible = [v for v in cache[r[\'id\']] if r[\'known_mask\'] & (1 << v[\'classe_id\'])\n                    and v[\'confianca\'] >= (min(protocol.GRID) if v[\'classe_id\'] in protocol.TUNED_IDS else .25)]\n        pred = core.with_policy(eligible, r[\'width\'], r[\'height\'], policy)\n        fixed = {f: 0 for f in protocol.FIELDS}\n        variable = {}\n        for cid in range(12):\n            gs = [g for g in gt if g[\'classe_id\'] == cid]\n            ps = [p for p in pred if p[\'classe_id\'] == cid]\n            if cid in protocol.TUNED_IDS:\n                variable[str(cid)] = {str(t): _counts(gs, [p for p in ps if p[\'confianca\'] >= t], matcher) for t in protocol.GRID}\n            else:\n                count = _counts(gs, ps, matcher)\n                for f in fixed:\n                    fixed[f] += count[f]\n        fixed[\'gt_ignorados\'] = len(r[\'gt\']) - len(gt)\n        book.append({\'id\': r[\'id\'], \'coorte\': r[\'coorte\'], \'partition\': \'calibracao\',\n                     \'fixed\': fixed, \'variable\': variable})\n        if progress and (index % 100 == 0 or index == len(records)):\n            progress(f\'Fronteira de calibração {r["coorte"]}: {index}/{len(records)}\')\n    return book\n\ndef frontier_summary(book, thresholds):\n    p = protocol.validate_profile(thresholds)\n    if not book or any(row[\'partition\'] != \'calibracao\' for row in book):\n        raise ValueError(\'Fronteira não pertence à calibração.\')\n    sums = Counter()\n    for row in book:\n        sums.update(row[\'fixed\'])\n        for cid in protocol.TUNED_IDS:\n            sums.update(row[\'variable\'][str(cid)][str(p[cid])])\n    return {\'coorte\': book[0][\'coorte\'], \'imagens\': len(book),\n            \'alertas_por_objeto\': core.alert_metrics(sums[\'tp_alerta\'], sums[\'fn_alerta\'], sums[\'fp_alerta\']),\n            \'alta_ref\': sums[\'alta_ref\'], \'alta_sem_alerta\': sums[\'alta_sem_alerta\'],\n            \'detector_no_limiar\': core.alert_metrics(sums[\'tp_detector\'], sums[\'fn_detector\'], sums[\'fp_detector\']),\n            \'gt_ignorados_sem_cobertura\': sums[\'gt_ignorados\'], \'partition\': \'calibracao\'}\n\ndef search_grid(books, rule):\n    if set(books) != set(protocol.COHORTS):\n        raise ValueError(\'São necessárias as duas coortes de calibração.\')\n    rows = []\n    for chair in protocol.GRID:\n        for table in protocol.GRID:\n            p = protocol.profile(chair, table)\n            home = frontier_summary(books[protocol.COHORTS[0]], p)\n            external = frontier_summary(books[protocol.COHORTS[1]], p)\n            rows.append({\'id\': protocol.profile_id(p), \'thresholds\': p, \'partition\': \'calibracao\',\n                         \'home\': home, \'external\': external, \'checks\': protocol.judge(home, external, rule)})\n    return rows\n\ndef compare_reference(current, old, cohort, model):\n    """Reproduz contadores por imagem e estados de cada objeto GT em 0,25."""\n    fields = [\'tp_alerta\', \'fn_alerta\', \'fp_alerta\', \'alta_ref\', \'alta_sem_alerta\',\n              \'tp_detector\', \'fn_detector\', \'fp_detector\', \'gt_avaliaveis\', \'gt_ignorados\', \'predicoes_ignoradas_conf\']\n    a = {x[\'id\']: x for x in old[\'images\']}\n    b = {x[\'id\']: x for x in current[\'images\']}\n    if a.keys() != b.keys():\n        raise ValueError(\'IDs da referência divergentes.\')\n    rows = []\n    for iid in sorted(a):\n        for field in fields:\n            ok = int(a[iid][field]) == int(b[iid][field])\n            rows.append({\'modelo\': model, \'coorte\': cohort, \'id\': iid, \'metrica\': field,\n                         \'valor_16\': int(a[iid][field]), \'valor_17\': int(b[iid][field]), \'igual\': ok})\n    def gtmap(events):\n        result = {}\n        for x in events:\n            box = x[\'bbox_ref\']\n            if not box:\n                continue\n            box = json.loads(box) if isinstance(box, str) else box\n            k = (x[\'id\'], int(x[\'classe_id\']), tuple(round(float(v), 6) for v in box))\n            if k in result:\n                raise ValueError(\'Objeto GT repetido.\')\n            result[k] = x\n        return result\n    ga, gb = gtmap(old[\'events\']), gtmap(current[\'events\'])\n    if ga.keys() != gb.keys():\n        raise ValueError(\'Objetos GT diferentes da referência.\')\n    def norm(v):\n        return {\'True\': True, \'False\': False}.get(v, v) if isinstance(v, str) else v\n    gt_rows = []\n    for k in sorted(ga):\n        ok = all(norm(ga[k][f]) == norm(gb[k][f]) for f in\n                 [\'evento\', \'prioridade_ref\', \'prioridade_pred\', \'ha_alerta_ref\', \'ha_alerta_pred\'])\n        gt_rows.append({\'modelo\': model, \'coorte\': cohort, \'id\': k[0], \'classe_id\': k[1],\n                        \'bbox_ref\': json.dumps(k[2]), \'igual\': ok})\n    status = {\'modelo\': model, \'coorte\': cohort, \'checagens_imagem\': len(rows),\n              \'divergencias_imagem\': sum(not x[\'igual\'] for x in rows), \'objetos_GT\': len(gt_rows),\n              \'divergencias_objetos\': sum(not x[\'igual\'] for x in gt_rows)}\n    status[\'reproduziu\'] = status[\'divergencias_imagem\'] == status[\'divergencias_objetos\'] == 0\n    return status, rows, gt_rows\n', 'lim17_inputs.py': '"""Autentica o pacote 16 e reutiliza caches completos, com inferência só se faltar."""\nfrom __future__ import annotations\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\nimport diag11_core as core\nimport conf14_evaluation as historical\nimport lim17_protocol as protocol\nfrom lim17_io import DigestMismatch\n\nMODELS = (\'baseline\', \'candidato_10\')\nPREDICT_ARGS = {\'conf\': .001, \'imgsz\': 640, \'rect\': True, \'iou\': .70,\n                \'max_det\': 300, \'half\': False, \'batch\': 1}\n\ndef rows(raw):\n    return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\n\ndef load_source16(path, expected, output=None):\n    if core.sha256_file(path) != expected[\'pacote_16\']:\n        raise RuntimeError(\'O pacote 16 difere do resultado recebido.\')\n    result = {\'references\': {}, \'cache_manifests\': {}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist()) != len(set(z.namelist())) or z.testzip() is not None:\n            raise RuntimeError(\'Pacote 16 com membros repetidos ou CRC inválido.\')\n        for name in z.namelist():\n            core.safe_member(name)\n        summary = json.loads(z.read(\'resumo_experimento_16.json\'))\n        cfg = json.loads(z.read(\'configuracao_experimento_16.json\'))\n        if (not summary[\'concluido\'] or summary[\'reserva_futura_inferida\'] or summary[\'test_home_inferido\']\n                or summary[\'test_oi20k_inferido\'] or cfg[\'context_key\'] != expected[\'contexto_16\']):\n            raise RuntimeError(\'Resultado 16 incompatível ou incompleto.\')\n        if cfg[\'plano\'][\'initial_checkpoint_sha256\'] != expected[\'candidato\']:\n            raise RuntimeError(\'Candidato inicial diferente do esperado.\')\n        for name, sha in cfg[\'plano\'][\'codigo_sha256\'].items():\n            if hashlib.sha256(z.read(\'codigo/\' + name)).hexdigest() != sha:\n                raise RuntimeError(\'Código alterado no pacote 16: \' + name)\n        rule_raw = z.read(\'regra_selecao_congelada_12.json\')\n        if hashlib.sha256(rule_raw).hexdigest() != expected[\'regra_12_sha256\']:\n            raise RuntimeError(\'Regra histórica alterada.\')\n        result[\'rule_raw\'], result[\'rule\'] = rule_raw, json.loads(rule_raw)\n        result[\'source_folder\'] = \'resultados/\' + cfg[\'experimento\']\n        if result[\'source_folder\'] != expected[\'pasta_16\']:\n            raise RuntimeError(\'Pasta do resultado 16 incompatível.\')\n        for model in MODELS:\n            result[\'references\'][model], result[\'cache_manifests\'][model] = {}, {}\n            for cohort in protocol.COHORTS:\n                prefix = \'referencias/\' + model + \'/\' + cohort + \'/\'\n                result[\'references\'][model][cohort] = {\n                    \'summary\': json.loads(z.read(prefix + \'metricas.json\')),\n                    \'images\': rows(z.read(prefix + \'alertas_por_imagem.csv\')),\n                    \'events\': rows(z.read(prefix + \'avaliacao_objetos.csv\'))}\n                meta = json.loads(z.read(prefix + \'cache/cache_manifesto.json\'))\n                key = \'baseline\' if model == \'baseline\' else \'candidato\'\n                request = meta[\'solicitacao\']\n                if (not meta[\'concluido\'] or request[\'peso_sha256\'] != expected[key]\n                        or request[\'parametros\'] != PREDICT_ARGS or request[\'coorte\'] != cohort):\n                    raise RuntimeError(\'Manifesto do cache 16 incompatível.\')\n                result[\'cache_manifests\'][model][cohort] = meta\n        if output:\n            target = Path(output)\n            target.mkdir(parents=True, exist_ok=True)\n            (target / \'regra_selecao_congelada_12.json\').write_bytes(rule_raw)\n            core.save_json(target / \'fonte_autenticada_16.json\', {\n                \'pacote_sha256\': expected[\'pacote_16\'], \'contexto_16\': cfg[\'context_key\'],\n                \'pasta_16\': result[\'source_folder\'], \'status_16\': summary[\'status\']})\n    return result\n\ndef signature(records):\n    return [{k: r[k] for k in [\'id\', \'sha256\', \'label_sha256\', \'width\', \'height\', \'known_mask\', \'role\']} for r in records]\n\ndef validate_source_records(cohorts, source):\n    if set(cohorts) != set(protocol.COHORTS):\n        raise ValueError(\'Coortes diferentes do protocolo.\')\n    for cohort, records in cohorts.items():\n        if len(records) != (456 if cohort == protocol.COHORTS[0] else 1000):\n            raise ValueError(\'Quantidade de imagens mudou: \' + cohort)\n        if len({r[\'id\'] for r in records}) != len(records):\n            raise ValueError(\'IDs repetidos.\')\n        for model in MODELS:\n            request = source[\'cache_manifests\'][model][cohort][\'solicitacao\']\n            actual = hashlib.sha256(core.canonical(signature(records)).encode()).hexdigest()\n            if actual != request[\'entradas_sha256\']:\n                raise RuntimeError(\'A ordem, os bytes, as dimensões ou a cobertura diferem do cache 16: \' + cohort)\n            if set(r[\'id\'] for r in records) != {r[\'id\'] for r in source[\'references\'][model][cohort][\'images\']}:\n                raise RuntimeError(\'IDs diferentes da referência 16.\')\n        expected_role = \'home_val\' if cohort == protocol.COHORTS[0] else \'val_nova\'\n        if any(r[\'role\'] != expected_role for r in records):\n            raise RuntimeError(\'Papel proibido na validação.\')\n    return True\n\ndef infer_frozen(weight, records, directory, expected_weight, predict_fn=None, model_factory=None):\n    """Sem model.train/fit: uma passagem de inferência no peso autenticado."""\n    if not records or any(r[\'role\'] not in {\'home_val\', \'val_nova\'} for r in records):\n        raise ValueError(\'Inferência em dado protegido ou vazio bloqueada.\')\n    if core.sha256_file(weight) != expected_weight:\n        raise RuntimeError(\'Peso de inferência diferente da referência.\')\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    environment = {\'inferencia\': \'callback_de_teste\'}\n    if predict_fn is None:\n        import importlib.metadata\n        import subprocess\n        import sys\n        try:\n            version = importlib.metadata.version(\'ultralytics\')\n        except importlib.metadata.PackageNotFoundError:\n            version = None\n        if version != \'8.4.150\':\n            subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'ultralytics==8.4.150\'], check=True)\n        import torch\n        import ultralytics\n        import cv2\n        if ultralytics.__version__ != \'8.4.150\':\n            raise RuntimeError(\'A inferência de recuperação exige ultralytics==8.4.150. Reinicie a sessão após a instalação.\')\n        from ultralytics import YOLO, settings\n        settings.update({k: False for k in [\'sync\', \'hub\', \'platform\', \'wandb\', \'comet\', \'mlflow\', \'neptune\', \'clearml\', \'dvc\', \'tensorboard\'] if k in settings})\n        torch.set_num_threads(2)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n        cv2.setNumThreads(0)\n        device = 0 if torch.cuda.is_available() else \'cpu\'\n        model = (model_factory or YOLO)(str(weight))\n        if [str(model.names[i]) for i in range(12)] != core.NAMES:\n            raise RuntimeError(\'Classes do peso incompatíveis.\')\n        for p in model.model.parameters():\n            p.requires_grad_(False)\n        model.model.eval()\n        predict_fn = lambda r: core.model_predictions(model, r, device)\n        environment = {\'ultralytics\': ultralytics.__version__, \'torch\': torch.__version__,\n                       \'device\': str(device), \'gpu\': torch.cuda.get_device_name(0) if device == 0 else None}\n    path = directory / \'predicoes_conf_0p001.jsonl.gz\'\n    tmp = path.with_suffix(\'.tmp\')\n    with gzip.open(tmp, \'wt\', encoding=\'utf-8\', compresslevel=4) as f:\n        for i, r in enumerate(records, 1):\n            if r[\'role\'] not in {\'home_val\', \'val_nova\'}:\n                raise ValueError(\'Inferência em dado protegido bloqueada.\')\n            preds = predict_fn(r)\n            historical.validate_predictions(preds, r)\n            f.write(json.dumps({\'id\': r[\'id\'], \'image_sha256\': r[\'sha256\'],\n                                \'label_sha256\': r[\'label_sha256\'], \'predicoes\': preds}, ensure_ascii=False) + \'\\n\')\n            if i % 100 == 0 or i == len(records):\n                print(f\'Inferência de recuperação {r["coorte"]}: {i}/{len(records)}\', flush=True)\n    tmp.replace(path)\n    meta = {\'source\': \'inferencia_17_sem_treino\', \'peso_sha256\': expected_weight,\n            \'entradas_sha256\': hashlib.sha256(core.canonical(signature(records)).encode()).hexdigest(),\n            \'parametros\': PREDICT_ARGS, \'ambiente_inferencia\': environment,\n            \'cache_sha256\': core.sha256_file(path), \'concluido\': True}\n    core.save_json(directory / \'cache_manifesto_17.json\', meta)\n    return path, meta\n\ndef obtain_cache(model, cohort, records, source, expected, drive, work, output, infer_fn=infer_frozen):\n    expected_role = \'home_val\' if cohort == protocol.COHORTS[0] else \'val_nova\'\n    if model not in MODELS or cohort not in protocol.COHORTS or not records or any(r[\'role\'] != expected_role for r in records):\n        raise ValueError(\'Modelo, papel ou coorte proibidos no cache.\')\n    work, output = Path(work), Path(output)\n    original = source[\'cache_manifests\'][model][cohort]\n    relative = source[\'source_folder\'] + \'/referencias/\' + model + \'/\' + cohort + \'/cache/predicoes_conf_0p001.jsonl.gz\'\n    local = work / \'caches_fontes\' / model / cohort / \'predicoes_conf_0p001.jsonl.gz\'\n    target = output / \'caches_recriados\' / model / cohort\n    try:\n        path = drive.fetch(relative, local, original[\'cache_sha256\'], optional=True)\n    except DigestMismatch as exc:\n        core.save_json(output / \'auditoria_cache\' / model / (cohort + \'_cache_fonte_rejeitado.json\'), {\'motivo\': str(exc), \'acao\': \'recriar por inferencia\'})\n        path = None\n    if path is not None:\n        meta = {\'source\': \'cache_autenticado_16\', \'cache_sha256\': original[\'cache_sha256\'],\n                \'source_request\': original[\'solicitacao\'], \'ambiente_atual_igual_ao_da_captura_exigido\': False,\n                \'motivo\': \'propostas congeladas por SHA; não ocorre inferência ao reutilizar\'}\n    else:\n        key = \'baseline\' if model == \'baseline\' else \'candidato\'\n        prior_path, prior_meta = target / \'predicoes_conf_0p001.jsonl.gz\', target / \'cache_manifesto_17.json\'\n        prior = json.loads(prior_meta.read_text()) if prior_meta.is_file() else {}\n        request_sha = hashlib.sha256(core.canonical(signature(records)).encode()).hexdigest()\n        if (prior_path.is_file() and prior.get(\'cache_sha256\') == core.sha256_file(prior_path)\n                and prior.get(\'peso_sha256\') == expected[key] and prior.get(\'entradas_sha256\') == request_sha\n                and prior.get(\'parametros\') == PREDICT_ARGS and prior.get(\'concluido\')):\n            path, meta = prior_path, prior\n        else:\n            relative_weight = (\'modelos/yolov8n_baseline/best.pt\' if model == \'baseline\'\n                               else expected[\'pasta_10\'] + \'/best.pt\')\n            weight = drive.fetch(relative_weight, work / \'pesos\' / (model + \'.pt\'), expected[key])\n            path, meta = infer_fn(weight, records, target, expected[key])\n        meta = {**meta, \'cache_16_original_sha256\': original[\'cache_sha256\']}\n    if not path.is_file() or core.sha256_file(path) != meta[\'cache_sha256\']:\n        raise RuntimeError(\'Cache indisponível ou alterado após recuperação.\')\n    cached = historical.read_cache(path, records)\n    core.save_json(output / \'auditoria_cache\' / model / (cohort + \'.json\'), meta)\n    print(\'Cache pronto:\', model, cohort, meta[\'source\'], flush=True)\n    return cached, meta\n', 'lim17_io.py': '"""Drive, recibos atômicos e pacote leve; credenciais ficam fora dos resultados."""\nfrom __future__ import annotations\nimport base64\nimport configparser\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport zipfile\n\nimport diag11_core as core\n\nclass DigestMismatch(RuntimeError):\n    pass\n\nclass Drive:\n    def __init__(self, binary, config, root=\'gdrive_tcc:TCC_Visao_Computacional\'):\n        self.binary, self.config, self.root = str(binary), str(config), root\n\n    @classmethod\n    def from_kaggle(cls):\n        from kaggle_secrets import UserSecretsClient\n        root = Path(\'/tmp/tcc_rclone\')\n        root.mkdir(parents=True, exist_ok=True, mode=0o700)\n        root.chmod(0o700)\n        try:\n            secret = UserSecretsClient().get_secret(\'GDRIVE_RCLONE_CONFIG_B64\')\n        except Exception:\n            raise RuntimeError(\'Habilite o Secret existente GDRIVE_RCLONE_CONFIG_B64 em Add-ons → Secrets.\') from None\n        try:\n            raw = base64.b64decode(secret.strip(), validate=True)\n            cfg = configparser.ConfigParser(interpolation=None)\n            cfg.read_string(raw.decode(\'utf-8-sig\'))\n        except Exception:\n            raise RuntimeError(\'A configuração do Secret rclone não é válida.\') from None\n        if (not cfg.has_section(\'gdrive_tcc\') or cfg.get(\'gdrive_tcc\', \'type\', fallback=\'\') != \'drive\'\n                or not cfg.get(\'gdrive_tcc\', \'client_id\', fallback=\'\')\n                or not cfg.get(\'gdrive_tcc\', \'token\', fallback=\'\')\n                or cfg.get(\'gdrive_tcc\', \'scope\', fallback=\'\') != \'drive\'):\n            raise RuntimeError(\'O Secret precisa da conexão gdrive_tcc autorizada para leitura e gravação em Meu Drive.\')\n        cfg.remove_option(\'gdrive_tcc\', \'team_drive\')\n        config = root / \'gdrive_tcc.conf\'\n        with config.open(\'w\', encoding=\'utf-8\') as f:\n            cfg.write(f)\n        config.chmod(0o600)\n        del secret, raw, cfg\n        binary = root / \'rclone\'\n        if not binary.is_file():\n            executable = shutil.which(\'rclone\')\n            if not executable:\n                import os\n                prefix = [] if os.geteuid() == 0 else [\'sudo\']\n                subprocess.run(prefix + [\'apt-get\', \'update\', \'-qq\'], check=True)\n                subprocess.run(prefix + [\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', \'rclone\'], check=True)\n                executable = shutil.which(\'rclone\')\n            if not executable:\n                raise RuntimeError(\'Executável rclone indisponível após a instalação.\')\n            shutil.copy2(executable, binary)\n            binary.chmod(0o700)\n        return cls(binary, config)\n\n    def _call(self, args):\n        return subprocess.run([self.binary, \'--config\', self.config] + list(args)\n                              + [\'--retries\', \'2\', \'--low-level-retries\', \'3\',\n                                 \'--contimeout\', \'15s\', \'--timeout\', \'60s\'],\n                              capture_output=True, text=True)\n\n    def exists(self, relative):\n        result = self._call([\'lsjson\', self.root + \'/\' + relative, \'--stat\'])\n        if result.returncode == 0:\n            return True\n        if result.returncode in {3, 4}:\n            return False\n        raise RuntimeError(\'Falha ao consultar Drive (código \' + str(result.returncode) + \'): \' + relative)\n\n    def fetch(self, relative, target, expected_sha=None, optional=False):\n        target = Path(target)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        receipt = target.with_name(target.name + \'.download.json\')\n        if target.is_file() and target.stat().st_size:\n            local_sha = core.sha256_file(target)\n            prior = {}\n            if receipt.is_file():\n                try:\n                    prior = json.loads(receipt.read_text())\n                except (ValueError, OSError):\n                    pass\n            if ((expected_sha and local_sha == expected_sha)\n                    or (not expected_sha and prior.get(\'origem\') == relative and prior.get(\'sha256\') == local_sha)):\n                print(\'Entrada local conferida:\', relative, flush=True)\n                return target\n        if optional and not self.exists(relative):\n            return None\n        print(\'Baixando:\', relative, flush=True)\n        tmp = target.with_name(target.name + \'.part\')\n        result = self._call([\'copyto\', self.root + \'/\' + relative, str(tmp)])\n        if result.returncode:\n            raise RuntimeError(\'Download interrompido (código \' + str(result.returncode) + \'): \' + relative)\n        if not tmp.is_file() or not tmp.stat().st_size:\n            raise RuntimeError(\'Download vazio: \' + relative)\n        actual = core.sha256_file(tmp)\n        if expected_sha and actual != expected_sha:\n            tmp.replace(target.with_name(target.name + \'.rejeitado_\' + actual[:12]))\n            raise DigestMismatch(\'SHA-256 diferente da fonte autenticada: \' + relative)\n        tmp.replace(target)\n        core.save_json(receipt, {\'origem\': relative, \'sha256\': actual, \'bytes\': target.stat().st_size})\n        return target\n\n    def backup(self, local, relative, required=True):\n        print(\'Salvando resultados no Drive...\', flush=True)\n        result = self._call([\'copy\', str(local), self.root + \'/\' + relative,\n                             \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\', \'--exclude\', \'**/__pycache__/**\'])\n        if result.returncode and required:\n            raise RuntimeError(\'Falha no salvamento no Drive; resultados locais preservados. Código \' + str(result.returncode))\n        return result.returncode == 0\n\n    def restore(self, relative, local):\n        if not self.exists(relative):\n            return False\n        result = self._call([\'copy\', self.root + \'/\' + relative, str(local),\n                             \'--exclude\', \'*.zip\', \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\'])\n        if result.returncode:\n            raise RuntimeError(\'Não foi possível recuperar o resultado anterior; código \' + str(result.returncode))\n        return True\n\ndef package(directory):\n    directory = Path(directory)\n    target = directory / \'pacote_calibracao_17.zip\'\n    tmp = target.with_suffix(\'.tmp\')\n    with zipfile.ZipFile(tmp, \'w\', zipfile.ZIP_DEFLATED, compresslevel=4) as z:\n        for path in sorted(directory.rglob(\'*\')):\n            if not path.is_file() or path in {target, tmp} or path.suffix in {\'.pt\', \'.tmp\', \'.part\'}:\n                continue\n            relative = path.relative_to(directory)\n            if any(p in {\'caches_recriados\', \'caches_fontes\', \'__pycache__\'} for p in relative.parts):\n                continue\n            z.write(path, relative.as_posix())\n    tmp.replace(target)\n    return target\n\ndef failure(directory, error, stage, drive=None, remote=None):\n    core.save_json(Path(directory) / \'execucao_interrompida.json\', {\n        \'concluido\': False, \'etapa\': stage, \'tipo\': type(error).__name__,\n        \'mensagem\': str(error)[:1400], \'modelo_substituido\': False,\n        \'treinamento_executado\': False, \'reserva_inferida\': False,\n        \'preenchimento_manual_obrigatorio\': False})\n    result = package(directory)\n    if drive and remote:\n        try:\n            drive.backup(directory, remote, required=False)\n        except Exception:\n            pass\n    return result\n', 'lim17_protocol.py': '"""Protocolo 17: dois limiares, uma escolha e uma verificação interna."""\nfrom __future__ import annotations\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nimport diag11_core as core\nimport diag11_data as data11\n\nVERSION = \'17-limiares-chair-table-1.0.0\'\nCOHORTS = (\'home_val_historica\', \'openimages_val_nova\')\nPARTITIONS = (\'calibracao\', \'verificacao_interna\')\nGRID = (.10, .15, .20, .25, .30, .35, .40)\nTUNED_IDS = (2, 3)\nFIXED = .25\nSPLIT_SEED = \'17-grupos-42\'\nMIN_IMAGES = 50\nMIN_ALERT_GT = 30\nFIELDS = (\'tp_alerta\', \'fn_alerta\', \'fp_alerta\', \'alta_ref\', \'alta_sem_alerta\',\n          \'tp_detector\', \'fn_detector\', \'fp_detector\', \'gt_avaliaveis\', \'gt_ignorados\')\nSPEC = {\'version\': VERSION, \'model\': \'candidato_10\', \'tuned_class_ids\': list(TUNED_IDS),\n        \'threshold_grid\': list(GRID), \'fixed_threshold_other_classes\': FIXED,\n        \'configurations\': 49, \'split_seed\': SPLIT_SEED,\n        \'split\': \'grupos SHA256/pHash64 distancia<=4; hash do grupo, sem previsoes\',\n        \'split_fraction_target\': .5, \'min_images_per_cohort_partition\': MIN_IMAGES,\n        \'min_alert_objects_per_tuned_class_cohort_partition\': MIN_ALERT_GT,\n        \'selection\': \'somente calibracao; requisitos relativos as referencias do mesmo subconjunto\',\n        \'ranking\': \'precisao Home; recall Home; menos altas omitidas Home; F1 externa; menos classes alteradas; distancia de 0.25; limiares\',\n        \'verification\': \'uma configuracao congelada; sem segunda escolha se falhar\',\n        \'full_validation\': \'descritiva, apos a escolha; seis limites numericos originais\',\n        \'preexposure\': \'ambas as validacoes ja influenciaram o desenvolvimento\',\n        \'training\': False, \'test_inference\': False, \'reserved_inference\': False}\n\ndef canonical(value):\n    return json.dumps(core.json_clean(value), sort_keys=True, ensure_ascii=False,\n                      separators=(\',\', \':\'), allow_nan=False)\n\ndef digest(value):\n    return hashlib.sha256(canonical(value).encode()).hexdigest()\n\ndef profile(chair=.25, table=.25):\n    if chair not in GRID or table not in GRID:\n        raise ValueError(\'Limiar fora da grade congelada.\')\n    return {cid: float(chair if cid == 2 else table if cid == 3 else FIXED)\n            for cid in range(len(core.NAMES))}\n\ndef validate_profile(thresholds):\n    normalized = {int(k): v for k, v in thresholds.items()}\n    if set(normalized) != set(range(12)):\n        raise ValueError(\'Perfil precisa das doze classes.\')\n    if normalized != profile(normalized[2], normalized[3]):\n        raise ValueError(\'Somente cadeira e mesa podem variar na grade congelada.\')\n    return normalized\n\ndef profile_id(thresholds):\n    p = validate_profile(thresholds)\n    return f\'chair_{p[2]:.2f}_table_{p[3]:.2f}\'.replace(\'.\', \'p\')\n\ndef group_split(cohorts, output=None):\n    """Componentes transitivos de imagens; nenhuma previsão entra na divisão."""\n    records = sorted([r for c in COHORTS for r in cohorts[c]], key=lambda r: (r[\'coorte\'], r[\'id\']))\n    if len({(r[\'coorte\'], r[\'id\']) for r in records}) != len(records):\n        raise ValueError(\'IDs repetidos na divisão.\')\n    parent = list(range(len(records)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n    def union(i, j):\n        a, b = find(i), find(j)\n        if a != b:\n            parent[max(a, b)] = min(a, b)\n    phashes = [(int(r[\'phash\'], 16) if isinstance(r[\'phash\'], str) else int(r[\'phash\']))\n               if \'phash\' in r else data11.phash(r[\'image\']) for r in records]\n    links = 0\n    for i, r in enumerate(records):\n        for j in range(i):\n            if r[\'sha256\'] == records[j][\'sha256\'] or (phashes[i] ^ phashes[j]).bit_count() <= 4:\n                union(i, j)\n                links += 1\n    components = defaultdict(list)\n    for i in range(len(records)):\n        components[find(i)].append(i)\n    rows = []\n    assigned = {p: {c: [] for c in COHORTS} for p in PARTITIONS}\n    for indices in components.values():\n        group = digest(sorted({records[i][\'sha256\'] for i in indices}))\n        bit = int(hashlib.sha256((SPLIT_SEED + \'|\' + group).encode()).hexdigest(), 16) % 2\n        partition = PARTITIONS[bit]\n        for i in indices:\n            r = {**records[i], \'partition\': partition, \'group\': group, \'phash\': phashes[i]}\n            assigned[partition][r[\'coorte\']].append(r)\n            rows.append({\'coorte\': r[\'coorte\'], \'id\': r[\'id\'], \'sha256\': r[\'sha256\'],\n                         \'phash\': f\'{phashes[i]:016x}\', \'group\': group, \'partition\': partition})\n    rows.sort(key=lambda r: (r[\'coorte\'], r[\'id\']))\n    for p in PARTITIONS:\n        for c in COHORTS:\n            assigned[p][c].sort(key=lambda r: r[\'id\'])\n    counts = {p: {c: len(assigned[p][c]) for c in COHORTS} for p in PARTITIONS}\n    audit = {\'version\': VERSION, \'seed\': SPLIT_SEED, \'group_count\': len(components),\n             \'similarity_links\': links, \'counts\': counts, \'split_sha256\': digest(rows),\n             \'uses_predictions\': False, \'verification_is_independent_final_test\': False}\n    if output:\n        core.save_csv(Path(output) / \'divisao_calibracao_verificacao.csv\', rows)\n        core.save_json(Path(output) / \'auditoria_divisao.json\', audit)\n    return assigned, rows, audit\n\ndef check_split_support(partitions, minimum_images=MIN_IMAGES, minimum_alert_gt=MIN_ALERT_GT, policy=None):\n    if policy is None:\n        raise ValueError(\'Política V2 necessária para conferir o suporte.\')\n    rows = []\n    for partition in PARTITIONS:\n        for cohort in COHORTS:\n            records = partitions[partition][cohort]\n            if len(records) < minimum_images:\n                raise RuntimeError(\'Divisão com poucas imagens: \' + partition + \'/\' + cohort)\n            supports = Counter()\n            for r in records:\n                for g in core.with_policy([g for g in r[\'gt\'] if r[\'known_mask\'] & (1 << g[\'classe_id\'])], r[\'width\'], r[\'height\'], policy):\n                    if core.alert_on(g):\n                        supports[g[\'classe_id\']] += 1\n            for cid in TUNED_IDS:\n                n = supports[cid]\n                rows.append({\'partition\': partition, \'coorte\': cohort, \'classe_id\': cid,\n                             \'classe\': core.NAMES[cid], \'objetos_alerta_ref\': n})\n                if n < minimum_alert_gt:\n                    raise RuntimeError(\'Suporte insuficiente; a divisão não será refeita: \' + str(rows[-1]))\n    return rows\n\ndef rule_from_references(baseline_home, candidate_external):\n    h, e = baseline_home[\'alertas_por_objeto\'], candidate_external[\'alertas_por_objeto\']\n    for s in [h, e]:\n        if any(s[k] is None or not math.isfinite(s[k]) for k in [\'precision\', \'recall\', \'f1\']):\n            raise ValueError(\'Referência sem suporte para os critérios.\')\n    return {\'home\': {\'precision_maior_que\': h[\'precision\'], \'recall_minimo\': h[\'recall\'],\n                     \'alta_sem_alerta_maximo\': baseline_home[\'alta_sem_alerta\']},\n            \'externa\': {\'precision_minima\': e[\'precision\'], \'recall_minimo\': e[\'recall\'],\n                        \'alta_sem_alerta_maximo\': candidate_external[\'alta_sem_alerta\']},\n            \'origem\': \'referencias em 0.25 nos mesmos IDs; regra proporcional e contagens locais\'}\n\ndef judge(home, external, rule):\n    h, e = home[\'alertas_por_objeto\'], external[\'alertas_por_objeto\']\n    valid = all(s[k] is not None and math.isfinite(s[k]) for s in [h, e] for k in [\'precision\', \'recall\', \'f1\'])\n    if not valid:\n        return {\'elegivel\': False, \'metricas_com_suporte\': False}\n    result = {\'home_precision_superior\': h[\'precision\'] > rule[\'home\'][\'precision_maior_que\'] + 1e-12,\n              \'home_recall_preservado\': h[\'recall\'] >= rule[\'home\'][\'recall_minimo\'] - 1e-12,\n              \'home_altas_preservadas\': home[\'alta_sem_alerta\'] <= rule[\'home\'][\'alta_sem_alerta_maximo\'],\n              \'externa_precision_preservada\': e[\'precision\'] >= rule[\'externa\'][\'precision_minima\'] - 1e-12,\n              \'externa_recall_preservado\': e[\'recall\'] >= rule[\'externa\'][\'recall_minimo\'] - 1e-12,\n              \'externa_altas_preservadas\': external[\'alta_sem_alerta\'] <= rule[\'externa\'][\'alta_sem_alerta_maximo\']}\n    return {**result, \'elegivel\': all(result.values()), \'metricas_com_suporte\': True}\n\ndef choose(calibration_grid):\n    if any(r[\'partition\'] != \'calibracao\' for r in calibration_grid):\n        raise ValueError(\'Seleção recebeu resultados de verificação.\')\n    if len(calibration_grid) != 49 or len({r[\'id\'] for r in calibration_grid}) != 49:\n        raise ValueError(\'A grade precisa de exatamente 49 configurações distintas.\')\n    good = [r for r in calibration_grid if r[\'checks\'][\'elegivel\']]\n    def key(r):\n        p = validate_profile(r[\'thresholds\'])\n        h, e = r[\'home\'][\'alertas_por_objeto\'], r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'], -h[\'recall\'], r[\'home\'][\'alta_sem_alerta\'], -e[\'f1\'],\n                sum(p[c] != .25 for c in TUNED_IDS), sum(abs(p[c] - .25) for c in TUNED_IDS), p[2], p[3])\n    selected = min(good, key=key) if good else None\n    return {\'selected\': selected, \'eligible_count\': len(good), \'configurations\': 49,\n            \'used_partition\': \'calibracao\', \'verification_used_for_selection\': False,\n            \'status\': \'configuracao_congelada_para_verificacao\' if selected else \'nenhuma_configuracao_elegivel_na_calibracao\'}\n\ndef immutable_json(path, value):\n    path = Path(path)\n    if path.is_file():\n        if digest(json.loads(path.read_text())) != digest(value):\n            raise RuntimeError(\'Registro congelado diverge; não será sobrescrito: \' + path.name)\n    else:\n        core.save_json(path, value)\n    return digest(value)\n', 'lim17_reports.py': '"""Relatório de seleção e verificação, sem promoção automática de modelo."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport base64\nimport html\nimport json\nfrom pathlib import Path\n\nimport diag11_core as core\nimport lim17_protocol as protocol\n\ndef grid_rows(grid):\n    rows = []\n    for r in grid:\n        h, e = r[\'home\'][\'alertas_por_objeto\'], r[\'external\'][\'alertas_por_objeto\']\n        p = protocol.validate_profile(r[\'thresholds\'])\n        rows.append({\'id\': r[\'id\'], \'partition\': \'calibracao\', \'limiar_cadeira\': p[2], \'limiar_mesa\': p[3],\n                     \'home_precision\': h[\'precision\'], \'home_recall\': h[\'recall\'], \'home_f1\': h[\'f1\'],\n                     \'home_tp\': h[\'tp\'], \'home_fn\': h[\'fn\'], \'home_fp\': h[\'fp\'],\n                     \'home_alta_sem_alerta\': r[\'home\'][\'alta_sem_alerta\'],\n                     \'externa_precision\': e[\'precision\'], \'externa_recall\': e[\'recall\'], \'externa_f1\': e[\'f1\'],\n                     \'externa_tp\': e[\'tp\'], \'externa_fn\': e[\'fn\'], \'externa_fp\': e[\'fp\'],\n                     \'externa_alta_sem_alerta\': r[\'external\'][\'alta_sem_alerta\'], **r[\'checks\']})\n    return rows\n\ndef _table(rows, columns):\n    if not rows:\n        return \'<p>Não há configuração selecionada para esta etapa.</p>\'\n    def cell(value):\n        if isinstance(value, float):\n            value = f\'{value:.5f}\'\n        return html.escape(str(value))\n    return \'<table><thead><tr>\' + \'\'.join(\'<th>\' + html.escape(label) + \'</th>\' for _, label in columns) + \'</tr></thead><tbody>\' + \'\'.join(\n        \'<tr>\' + \'\'.join(\'<td>\' + cell(row.get(key, \'\')) + \'</td>\' for key, _ in columns) + \'</tr>\' for row in rows) + \'</tbody></table>\'\n\ndef plot_calibration(grid, rule, selection, output):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)\n    selected = selection[\'selected\']\n    for ax, key, label, rule_key in zip(axes, [\'home\', \'external\'], [\'Home\', \'Open Images externa\'], [\'home\', \'externa\']):\n        for r in grid:\n            a = r[key][\'alertas_por_objeto\']\n            ax.scatter(100 * a[\'recall\'], 100 * a[\'precision\'], s=28,\n                       color=\'#176b54\' if r[\'checks\'][\'elegivel\'] else \'#adb5bd\', alpha=.8)\n        if selected:\n            a = selected[key][\'alertas_por_objeto\']\n            ax.scatter(100 * a[\'recall\'], 100 * a[\'precision\'], marker=\'*\', s=180, color=\'#b65e00\', label=\'Escolha congelada\')\n            ax.legend(loc=\'best\')\n        limits = rule[rule_key]\n        ax.axvline(100 * limits[\'recall_minimo\'], color=\'#983b35\', linestyle=\'--\', linewidth=1)\n        ax.axhline(100 * limits.get(\'precision_maior_que\', limits.get(\'precision_minima\')), color=\'#983b35\', linestyle=\'--\', linewidth=1)\n        ax.set(xlabel=\'Recall de alertas (%)\', ylabel=\'Precisão de alertas (%)\', title=label + \' — calibração\')\n        ax.grid(alpha=.18)\n    fig.suptitle(\'49 configurações; as linhas mostram requisitos locais de precisão e recall\')\n    target = Path(output) / \'grade_calibracao.png\'\n    fig.savefig(target, dpi=150)\n    plt.close(fig)\n    return target\n\ndef write_report(output, summary, grid, rules, selection, comparisons):\n    output = Path(output)\n    core.save_json(output / \'resumo_calibracao_17.json\', summary)\n    core.save_csv(output / \'grade_calibracao.csv\', grid_rows(grid))\n    core.save_csv(output / \'comparacao_referencias_e_configuracao.csv\', comparisons)\n    failures = Counter(key for row in grid for key, value in row[\'checks\'].items()\n                       if key not in {\'elegivel\', \'metricas_com_suporte\'} and value is False)\n    core.save_json(output / \'criterios_nao_atendidos_na_grade.json\', dict(failures))\n    picture = plot_calibration(grid, rules[\'calibracao\'], selection, output)\n    image = base64.b64encode(picture.read_bytes()).decode()\n    selected = selection[\'selected\']\n    threshold_text = \'Nenhuma configuração elegível na calibração.\'\n    if selected:\n        p = protocol.validate_profile(selected[\'thresholds\'])\n        threshold_text = f\'Candidato 10: cadeira {p[2]:.2f}; mesa {p[3]:.2f}; demais classes 0,25.\'\n    text = \'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><title>Resultado 17</title><style>\'\n    text += \'body{font:16px system-ui,sans-serif;color:#202a30;max-width:1100px;margin:32px auto;padding:0 20px;line-height:1.55}table{border-collapse:collapse;font-size:14px;width:100%}td,th{padding:8px;border-bottom:1px solid #d9e0e3;text-align:left}th{background:#edf3f2}img{width:100%;height:auto}code{background:#edf2f4;padding:2px 5px}.status{background:#edf3f2;padding:16px;border-left:4px solid #176b54}</style>\'\n    text += \'<h1>Experimento 17 — limiares por classe</h1><p class="status">\' + html.escape(summary[\'mensagem\']) + \'</p>\'\n    text += \'<p>\' + html.escape(threshold_text) + \'</p><p>Não houve treino, alteração de rótulos ou substituição automática de modelo.</p>\'\n    text += \'<h2>Seleção e verificação</h2><p>A grade foi avaliada somente em calibração. Uma escolha, quando existente, foi congelada antes da verificação interna. Falhar não autoriza tentar a segunda colocada. As duas validações já participaram do desenvolvimento; a verificação não é teste final independente.</p>\'\n    text += \'<p>Configurações elegíveis em calibração: \' + str(selection[\'eligible_count\']) + \' de 49.</p>\'\n    text += \'<img alt="Precisão e recall das 49 configurações somente na calibração" src="data:image/png;base64,\' + image + \'">\'\n    text += \'<h2>Referências e configuração congelada</h2><p>Precisão e recall abaixo são proporções. O conjunto completo é descritivo e não foi usado para trocar a escolha.</p>\'\n    text += _table(comparisons, [(\'partition\',\'Partição\'), (\'modelo\',\'Modelo\'), (\'coorte\',\'Coorte\'),\n                              (\'precision\',\'Precisão\'), (\'recall\',\'Recall\'), (\'f1\',\'F1\'), (\'alta_sem_alerta\',\'Altas omitidas\')])\n    text += \'<h2>Critérios da escolha</h2>\' + \'<pre>\' + html.escape(json.dumps(summary.get(\'avaliacao_escolha\'), ensure_ascii=False, indent=2)) + \'</pre>\'\n    text += \'<h2>Limites</h2><ul>\' + \'\'.join(\'<li>\' + html.escape(s) + \'</li>\' for s in summary[\'limites\']) + \'</ul>\'\n    text += \'<p>O pacote contém os registros GT de validação, previsões compactas suficientes para esta grade, código e diários de decisão. Os caches completos e pesos ficam separados no Drive.</p>\'\n    text += \'<p>Resultados em <code>pacote_calibracao_17.zip</code>. Não há formulário a preencher.</p></html>\'\n    (output / \'relatorio_calibracao_17.html\').write_text(text, encoding=\'utf-8\')\n    (output / \'LEIA_RESULTADO.md\').write_text(\'# Experimento 17\\n\\n\' + summary[\'mensagem\'] + \'\\n\\n\' + threshold_text + \'\\n\\nResultados em pacote_calibracao_17.zip. A verificação é interna e já houve exposição prévia às validações. Nenhum modelo foi substituído.\\n\', encoding=\'utf-8\')\n\ndef comparison_row(partition, model, cohort, summary):\n    return {\'partition\': partition, \'modelo\': model, \'coorte\': cohort,\n            **summary[\'alertas_por_objeto\'], \'alta_ref\': summary[\'alta_ref\'],\n            \'alta_sem_alerta\': summary[\'alta_sem_alerta\']}\n', 'lim17_run.py': '"""Ordem do experimento: reproduzir, escolher em calibração, congelar, verificar."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport json\nfrom pathlib import Path\n\nimport diag11_core as core\nimport conf14_evaluation as historical\nimport lim17_protocol as protocol\nimport lim17_evaluation as evaluation\nimport lim17_reports as reports\nimport lim17_io as io17\n\nLIMITS = [\n    \'Calibração e verificação são partes das validações já usadas no desenvolvimento; não constituem teste final independente.\',\n    \'Ajuste restrito aos limiares de cadeira e mesa, uma grade de 49 opções e uma divisão agrupada; não garante melhora fora desses dados.\',\n    \'As referências e os rótulos são preservados; diferenças semânticas existentes na avaliação permanecem.\',\n    \'Propostas pós-NMS em confiança 0,001 e max_det=300; não há acesso a logits nem a propostas descartadas antes do cache.\',\n    \'Alertas medem concordância com a política V2; não validam risco físico, distância ou uso assistivo real.\',\n    \'Origem do treino original do baseline permanece não determinada.\',\n    \'Nenhum modelo ou configuração em uso é substituído automaticamente.\']\n\ndef save_gz(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.tmp\')\n    with tmp.open(\'wb\') as f, gzip.GzipFile(fileobj=f, mode=\'wb\', mtime=0, compresslevel=4) as z:\n        z.write(protocol.canonical(value).encode())\n    tmp.replace(path)\n\ndef _subset(result, ids, partition):\n    images = [{**x, \'partition\': partition} for x in result[\'images\'] if x[\'id\'] in ids]\n    events = [x for x in result[\'events\'] if x[\'id\'] in ids]\n    summary, classes = historical.summarise(images, events, .25)\n    summary.update({\'modelo\': result[\'summary\'][\'modelo\'], \'partition\': partition})\n    return {\'summary\': summary, \'classes\': classes, \'images\': images, \'events\': events}\n\ndef _save_result(path, result):\n    core.save_json(path / \'metricas.json\', result[\'summary\'])\n    core.save_csv(path / \'metricas_por_classe.csv\', result[\'classes\'])\n    core.save_csv(path / \'alertas_por_imagem.csv\', result[\'images\'])\n    core.save_csv(path / \'avaliacao_objetos.csv\', result[\'events\'])\n\ndef _same_counts(a, b):\n    fields = [\'tp\', \'fn\', \'fp\']\n    return (all(a[\'alertas_por_objeto\'][f] == b[\'alertas_por_objeto\'][f] for f in fields)\n            and a[\'alta_sem_alerta\'] == b[\'alta_sem_alerta\'] and a[\'alta_ref\'] == b[\'alta_ref\']\n            and all(a[\'detector_no_limiar\'][f] == b[\'detector_no_limiar\'][f] for f in fields)\n            and a[\'gt_ignorados_sem_cobertura\'] == b[\'gt_ignorados_sem_cobertura\'])\n\ndef run(cohorts, caches, source, partitions, split_rows, split_audit, policy, matcher,\n        output, context_key, backup=lambda stage: None):\n    """A verificação recebe uma única configuração, nunca a grade de candidatos."""\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if set(caches) != {\'baseline\', \'candidato_10\'}:\n        raise ValueError(\'Somente baseline e candidato 10 congelados pertencem ao protocolo.\')\n    for partition in protocol.PARTITIONS:\n        for cohort in protocol.COHORTS:\n            if any(r[\'partition\'] != partition for r in partitions[partition][cohort]):\n                raise ValueError(\'Partição declarada de forma inconsistente.\')\n    if protocol.digest(split_rows) != split_audit[\'split_sha256\']:\n        raise RuntimeError(\'A divisão mudou após ser registrada.\')\n    for cohort in protocol.COHORTS:\n        expected_ids = {r[\'id\'] for r in cohorts[cohort]}\n        sets = [{r[\'id\'] for r in partitions[p][cohort]} for p in protocol.PARTITIONS]\n        if sets[0] & sets[1] or sets[0] | sets[1] != expected_ids:\n            raise ValueError(\'Calibração e verificação precisam ser disjuntas e cobrir a coorte.\')\n\n    refs, reproduction, image_checks, gt_checks = {}, [], [], []\n    for model in [\'baseline\', \'candidato_10\']:\n        refs[model] = {}\n        for cohort in protocol.COHORTS:\n            result = evaluation.evaluate(cohorts[cohort], caches[model][cohort], protocol.profile(),\n                                         policy, matcher, model, output / \'referencias_025\' / model / cohort)\n            refs[model][cohort] = result\n            status, images, objects = evaluation.compare_reference(result, source[\'references\'][model][cohort], cohort, model)\n            reproduction.append(status)\n            image_checks.extend(images)\n            gt_checks.extend(objects)\n    core.save_csv(output / \'reproducao_025_por_imagem.csv\', image_checks)\n    core.save_csv(output / \'reproducao_025_por_objeto.csv\', gt_checks)\n    core.save_json(output / \'reproducao_025_resumo.json\', reproduction)\n    if not all(r[\'reproduziu\'] for r in reproduction):\n        raise RuntimeError(\'A referência em 0,25 não foi reproduzida. A busca não será iniciada.\')\n\n    rules = {\'completa_descritiva\': source[\'rule\']}\n    subset_refs = {}\n    comparisons = []\n    for partition in protocol.PARTITIONS:\n        subset_refs[partition] = {m: {} for m in refs}\n        for model in refs:\n            for cohort in protocol.COHORTS:\n                ids = {r[\'id\'] for r in partitions[partition][cohort]}\n                result = _subset(refs[model][cohort], ids, partition)\n                subset_refs[partition][model][cohort] = result\n                comparisons.append(reports.comparison_row(partition, model, cohort, result[\'summary\']))\n        rules[partition] = protocol.rule_from_references(\n            subset_refs[partition][\'baseline\'][protocol.COHORTS[0]][\'summary\'],\n            subset_refs[partition][\'candidato_10\'][protocol.COHORTS[1]][\'summary\'])\n    for model in refs:\n        for cohort in protocol.COHORTS:\n            comparisons.append(reports.comparison_row(\'completa_descritiva\', model, cohort, refs[model][cohort][\'summary\']))\n    protocol.immutable_json(output / \'regras_por_particao.json\', rules)\n    backup(\'referências reproduzidas e regras congeladas\')\n\n    books, frontier_checks = {}, []\n    for cohort in protocol.COHORTS:\n        records = partitions[\'calibracao\'][cohort]\n        path = output / \'fronteiras_calibracao\' / (cohort + \'.json.gz\')\n        marker = path.with_suffix(\'.concluido.json\')\n        request = {\'context_key\': context_key, \'ids\': [r[\'id\'] for r in records],\n                   \'partition\': \'calibracao\', \'grid\': list(protocol.GRID)}\n        meta = json.loads(marker.read_text()) if marker.is_file() else {}\n        if path.is_file() and meta.get(\'request\') == request and meta.get(\'sha256\') == core.sha256_file(path):\n            with gzip.open(path, \'rt\', encoding=\'utf-8\') as f:\n                books[cohort] = json.load(f)\n            print(\'Fronteira de calibração recuperada:\', cohort, flush=True)\n        else:\n            books[cohort] = evaluation.calibration_frontier(records, caches[\'candidato_10\'][cohort], policy, matcher,\n                                                           progress=lambda message: print(message, flush=True))\n            save_gz(path, books[cohort])\n            core.save_json(marker, {\'request\': request, \'sha256\': core.sha256_file(path)})\n        expected = subset_refs[\'calibracao\'][\'candidato_10\'][cohort][\'summary\']\n        actual = evaluation.frontier_summary(books[cohort], protocol.profile())\n        ok = _same_counts(expected, actual)\n        frontier_checks.append({\'coorte\': cohort, \'perfil\': \'uniforme_025\', \'contagens_iguais\': ok})\n        if not ok:\n            raise RuntimeError(\'O cálculo por classe não reproduziu a avaliação integral em calibração.\')\n        backup(\'fronteira calculada: \' + cohort)\n    core.save_json(output / \'auditoria_fronteira_025.json\', frontier_checks)\n    grid = evaluation.search_grid(books, rules[\'calibracao\'])\n    selection = protocol.choose(grid)\n    choice_record = {\'context_key\': context_key, \'split_sha256\': split_audit[\'split_sha256\'],\n                     \'grid_sha256\': protocol.digest(grid), \'selection\': selection,\n                     \'verification_configurations_allowed\': 1 if selection[\'selected\'] else 0,\n                     \'verification_result_used_for_selection\': False}\n    choice_sha = protocol.immutable_json(output / \'escolha_congelada_antes_verificacao.json\', choice_record)\n    core.save_csv(output / \'grade_calibracao.csv\', reports.grid_rows(grid))\n    # O backup precisa terminar antes de qualquer avaliação com limiar novo na verificação.\n    backup(\'escolha congelada ANTES da verificação\')\n    if protocol.digest(json.loads((output / \'escolha_congelada_antes_verificacao.json\').read_text())) != choice_sha:\n        raise RuntimeError(\'Escolha alterada depois do congelamento.\')\n\n    verification = full_checks = None\n    selected = selection[\'selected\']\n    status = \'nenhuma_configuracao_elegivel_na_calibracao\'\n    message = \'Nenhuma das 49 configurações cumpriu os requisitos na calibração. Nenhuma configuração nova foi avaliada na verificação.\'\n    if selected:\n        thresholds = protocol.validate_profile(selected[\'thresholds\'])\n        selected_results = {p: {} for p in protocol.PARTITIONS}\n        for cohort in protocol.COHORTS:\n            result = evaluation.evaluate(partitions[\'calibracao\'][cohort], caches[\'candidato_10\'][cohort], thresholds,\n                                         policy, matcher, selected[\'id\'], output / \'escolha\' / \'calibracao\' / cohort)\n            expected = selected[\'home\' if cohort == protocol.COHORTS[0] else \'external\']\n            if not _same_counts(expected, result[\'summary\']):\n                raise RuntimeError(\'A escolha recalculada integralmente diverge da grade; verificação bloqueada.\')\n            selected_results[\'calibracao\'][cohort] = result\n        for cohort in protocol.COHORTS:\n            result = evaluation.evaluate(partitions[\'verificacao_interna\'][cohort], caches[\'candidato_10\'][cohort], thresholds,\n                                         policy, matcher, selected[\'id\'], output / \'escolha\' / \'verificacao_interna\' / cohort)\n            selected_results[\'verificacao_interna\'][cohort] = result\n        verification = protocol.judge(selected_results[\'verificacao_interna\'][protocol.COHORTS[0]][\'summary\'],\n                                      selected_results[\'verificacao_interna\'][protocol.COHORTS[1]][\'summary\'], rules[\'verificacao_interna\'])\n        protocol.immutable_json(output / \'verificacao_unica.json\', {\n            \'context_key\': context_key, \'choice_sha256\': choice_sha, \'profile_id\': selected[\'id\'],\n            \'thresholds\': thresholds, \'distinct_configurations_evaluated\': 1, \'second_choice_attempts\': 0,\n            \'checks\': verification})\n        backup(\'verificação única concluída; sem nova seleção\')\n        full = {}\n        for partition in protocol.PARTITIONS:\n            for cohort in protocol.COHORTS:\n                comparisons.append(reports.comparison_row(partition, selected[\'id\'], cohort, selected_results[partition][cohort][\'summary\']))\n        for cohort in protocol.COHORTS:\n            im = [{**x} for p in protocol.PARTITIONS for x in selected_results[p][cohort][\'images\']]\n            events = [x for p in protocol.PARTITIONS for x in selected_results[p][cohort][\'events\']]\n            summary, classes = historical.summarise(im, events, .25)\n            summary.update({\'modelo\': selected[\'id\'], \'partition\': \'completa_descritiva\',\n                            \'conf_alertas\': .25 if thresholds == protocol.profile() else None,\n                            \'conf_alertas_por_classe\': thresholds, \'ap_recalculada\': False})\n            for row in classes:\n                row[\'conf_alertas\'] = thresholds[row[\'classe_id\']]\n            for row in im:\n                row[\'partition\'] = \'completa_descritiva\'\n            full[cohort] = {\'summary\': summary, \'classes\': classes, \'images\': im, \'events\': events}\n            _save_result(output / \'escolha\' / \'completa_descritiva\' / cohort, full[cohort])\n            comparisons.append(reports.comparison_row(\'completa_descritiva\', selected[\'id\'], cohort, summary))\n        full_checks = protocol.judge(full[protocol.COHORTS[0]][\'summary\'], full[protocol.COHORTS[1]][\'summary\'], source[\'rule\'])\n        if not verification[\'elegivel\']:\n            status = \'configuracao_nao_confirmada_na_verificacao\'\n            message = \'A configuração escolhida em calibração falhou na verificação interna. Nenhuma segunda configuração foi tentada; o baseline permanece como referência.\'\n        elif not full_checks[\'elegivel\']:\n            status = \'configuracao_nao_cumpre_limites_globais\'\n            message = \'A configuração passou na comparação local de verificação, mas não cumpriu todos os seis limites numéricos históricos na validação completa. Nenhum modelo foi substituído.\'\n        else:\n            status = \'configuracao_promissora_em_verificacao_interna\'\n            message = \'A configuração passou na verificação interna e nos seis limites numéricos históricos. É candidata para avaliação final independente; nenhum modelo foi substituído.\'\n        core.save_json(output / \'configuracao_inferencia_candidata.json\', {\n            \'model\': \'candidato_10\', \'thresholds\': thresholds, \'profile_id\': selected[\'id\'],\n            \'context_key\': context_key, \'aprovada_no_protocolo_interno\': verification[\'elegivel\'] and full_checks[\'elegivel\'],\n            \'implantada\': False, \'treinamento_executado\': False, \'uso\': \'filtrar propostas pós-NMS e recalcular correspondências e política V2\'})\n    summary = {\'version\': protocol.VERSION, \'concluido\': True, \'status\': status, \'mensagem\': message,\n               \'context_key\': context_key, \'primary_endpoint\': \'uma_escolha_calibracao_vs_referencias_na_verificacao_interna\',\n               \'configurations_evaluated_calibration\': 49, \'eligible_calibration\': selection[\'eligible_count\'],\n               \'distinct_configurations_evaluated_verification\': 1 if selected else 0,\n               \'selected_profile_id\': selected[\'id\'] if selected else None,\n               \'selected_thresholds\': selected[\'thresholds\'] if selected else None,\n               \'avaliacao_escolha\': {\'verificacao_interna\': verification, \'limites_globais_descritivos\': full_checks},\n               \'split\': split_audit, \'training_executed\': False, \'weights_changed\': False,\n               \'labels_changed\': False, \'model_replaced\': False, \'test_home_inferred\': False,\n               \'test_oi20k_inferred\': False, \'future_reserve_extracted\': False, \'future_reserve_inferred\': False,\n               \'verification_is_independent_final_test\': False, \'manual_form_required\': False,\n               \'source16_protocol_changed\': False, \'source16_reproduction_passed\': all(x[\'reproduziu\'] for x in reproduction),\n               \'limites\': LIMITS}\n    reports.write_report(output, summary, grid, rules, selection, comparisons)\n    core.save_json(output / \'integridade_final_17.json\', {\n        \'context_key\': context_key, \'choice_sha256\': choice_sha, \'split_sha256\': split_audit[\'split_sha256\'],\n        \'grade_usou_somente_calibracao\': True, \'verificacao_usou_no_maximo_uma_configuracao\': True,\n        \'reproducao_025_conferida\': summary[\'source16_reproduction_passed\'],\n        \'rotulos_e_pesos_preservados\': True, \'testes_e_reserva_inferidos\': False})\n    (output / \'execucao_interrompida.json\').unlink(missing_ok=True)\n    package = io17.package(output)\n    backup(\'resultado e pacote final\')\n    return summary, package\n', 'src/__init__.py': '', 'src/politica_alertas.py': '"""Parâmetros e funções da V2 original, sem reestimar com o TEST."""\nimport math\n\nNOMES_PT = {"bed": "cama", "sofa": "sofá", "chair": "cadeira", "table": "mesa",\n            "wardrobe": "guarda-roupa", "potted plant": "planta em vaso"}\nVERBOS_PT = {"bed": "detectada", "sofa": "detectado", "chair": "detectada", "table": "detectada",\n             "wardrobe": "detectado", "potted plant": "detectada"}\nORDEM_PRIORIDADE = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef avaliar_bbox(classe, bbox, largura, altura, versao="v2"):\n    if versao not in {"v1", "v2"}:\n        raise ValueError("Versão deve ser v1 ou v2.")\n    x1, y1, x2, y2 = map(float, bbox)\n    if largura <= 0 or altura <= 0 or not all(math.isfinite(v) for v in [x1, y1, x2, y2]):\n        raise ValueError("Dimensões ou coordenadas inválidas.")\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("Caixa sem área positiva.")\n    posicao = classificar_posicao((x1 + x2) / 2, largura)\n    tamanho, area = classificar_tamanho((x2 - x1) * (y2 - y1), largura * altura)\n    vertical, base = classificar_posicao_vertical(y2, altura)\n    resultado = (classificar_risco_v2(classe, posicao, tamanho, vertical) if versao == "v2"\n                 else classificar_risco_v1(classe, posicao, tamanho))\n    nivel = resultado["nivel_risco"]\n    return {"prioridade": nivel if nivel in {"baixo", "medio", "alto"} else "sem_prioridade",\n            "status_original": nivel, "pontuacao": resultado["pontuacao"],\n            "posicao": posicao, "tamanho": tamanho, "regiao_vertical": vertical,\n            "area_relativa": area, "base_vertical_relativa": base,\n            "ajuste_vertical": resultado.get("ajuste_vertical", 0)}\n\n\ndef gerar_mensagem(objeto, nivel_minimo="medio"):\n    if nivel_minimo not in {"medio", "alto"}:\n        raise ValueError("Nível mínimo deve ser medio ou alto.")\n    prioridade = objeto["prioridade"]\n    if ORDEM_PRIORIDADE.get(prioridade, 0) < ORDEM_PRIORIDADE[nivel_minimo]:\n        return ""\n    posicao = {"centro": "central", "esquerda": "esquerda", "direita": "direita"}[objeto["posicao"]]\n    classe = objeto["classe"]\n    return (f"Atenção: {NOMES_PT[classe]} {VERBOS_PT[classe]} na região {posicao} da imagem. "\n            f"Prioridade {\'alta\' if prioridade == \'alto\' else \'média\'}.")\n\n\ndef gerar_mensagens_tabela(df, coluna="nivel_risco_v2"):\n    if df.empty:\n        return []\n    candidatos = df.loc[df[coluna].isin(["medio", "alto"])].copy()\n    candidatos["_ordem"] = candidatos[coluna].map(ORDEM_PRIORIDADE)\n    candidatos = candidatos.sort_values(["_ordem", "ocupacao_imagem"], ascending=[False, False])\n    return [gerar_mensagem({"classe": r["classe"], "posicao": r["posicao"], "prioridade": r[coluna]})\n            for _, r in candidatos.iterrows()]\n\nPESO_OBJETO = {\n    "bed": 3,\n    "sofa": 3,\n    "chair": 3,\n    "table": 3,\n    "wardrobe": 3,\n    "potted plant": 2,\n}\n\nCLASSES_CONTEXTUAIS = {\n    "lamp",\n    "door",\n}\n\nCLASSES_IGNORADAS = {\n    "tv",\n    "laptop",\n    "window",\n    "photo frame",\n}\n\n\nLIMITE_ESQUERDA = 1 / 3\nLIMITE_DIREITA = 2 / 3\n\nLIMITE_AREA_MEDIA = 0.05\nLIMITE_AREA_GRANDE = 0.20\n\nLIMITE_VERTICAL_SUPERIOR = 0.60\nLIMITE_VERTICAL_INFERIOR = 0.80\n\nPESO_POSICAO = {\n    "esquerda": 1,\n    "centro": 3,\n    "direita": 1,\n}\n\nPESO_TAMANHO = {\n    "pequeno": 1,\n    "medio": 2,\n    "grande": 3,\n}\n\nAJUSTE_VERTICAL = {\n    "superior": -2,\n    "intermediaria": -1,\n    "inferior": 0,\n}\n\ndef classificar_posicao(\n    centro_x: float,\n    largura_imagem: int,\n) -> str:\n    centro_normalizado = centro_x / largura_imagem\n\n    if centro_normalizado < LIMITE_ESQUERDA:\n        return "esquerda"\n\n    if centro_normalizado < LIMITE_DIREITA:\n        return "centro"\n\n    return "direita"\n\n\ndef classificar_tamanho(\n    area_box: float,\n    area_imagem: float,\n):\n    proporcao = area_box / area_imagem\n\n    if proporcao < LIMITE_AREA_MEDIA:\n        return "pequeno", proporcao\n\n    if proporcao < LIMITE_AREA_GRANDE:\n        return "medio", proporcao\n\n    return "grande", proporcao\n\n\ndef classificar_posicao_vertical(\n    y2: float,\n    altura_imagem: int,\n):\n    proporcao_vertical = y2 / altura_imagem\n\n    if proporcao_vertical < LIMITE_VERTICAL_SUPERIOR:\n        return "superior", proporcao_vertical\n\n    if proporcao_vertical < LIMITE_VERTICAL_INFERIOR:\n        return "intermediaria", proporcao_vertical\n\n    return "inferior", proporcao_vertical\n\ndef classificar_risco_v1(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n):\n    if classe in CLASSES_IGNORADAS:\n        return {\n            "pontuacao": 0,\n            "nivel_risco": "ignorado",\n            "motivo": "classe sem relevância direta para locomoção",\n        }\n\n    if classe in CLASSES_CONTEXTUAIS:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "contextual",\n            "motivo": "classe exige informação adicional de contexto",\n        }\n\n    if classe not in PESO_OBJETO:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "nao_mapeado",\n            "motivo": "classe não configurada",\n        }\n\n    pontuacao = (\n        PESO_OBJETO[classe]\n        + PESO_POSICAO[posicao]\n        + PESO_TAMANHO[tamanho]\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "motivo": (\n            f"{classe} + posição {posicao} + tamanho {tamanho}"\n        ),\n    }\n\ndef classificar_risco_v2(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n    posicao_vertical: str,\n):\n    avaliacao_v1 = classificar_risco_v1(\n        classe=classe,\n        posicao=posicao,\n        tamanho=tamanho,\n    )\n\n    if avaliacao_v1["nivel_risco"] in {\n        "ignorado",\n        "contextual",\n        "nao_mapeado",\n    }:\n        return {\n            "pontuacao": avaliacao_v1["pontuacao"],\n            "nivel_risco": avaliacao_v1["nivel_risco"],\n            "ajuste_vertical": 0,\n            "motivo": avaliacao_v1["motivo"],\n        }\n\n    ajuste_vertical = AJUSTE_VERTICAL[posicao_vertical]\n\n    pontuacao = max(\n        1,\n        avaliacao_v1["pontuacao"] + ajuste_vertical,\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "ajuste_vertical": ajuste_vertical,\n        "motivo": (\n            f"{classe} + posição {posicao} "\n            f"+ tamanho {tamanho} "\n            f"+ região vertical {posicao_vertical}"\n        ),\n    }\n', 'src/validacao_alertas.py': '"""Compara detectores na validação Home, com a V2 e os parâmetros do 05d."""\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .politica_alertas import avaliar_bbox\n\nIMGSZ = 640\nCONF_ALERTA = 0.25\nIOU_NMS = 0.70\nIOU_CORRESPONDENCIA = 0.50\nMAX_DETECCOES = 300\n\n\ndef ler_objetos_yolo(caminho, largura, altura, nomes):\n    objetos = []\n    for n, linha in enumerate(Path(caminho).read_text(encoding="utf-8-sig").splitlines(), 1):\n        if not linha.strip():\n            continue\n        partes = linha.split()\n        if len(partes) != 5:\n            raise ValueError(f"Label inválido: {caminho}, linha {n}")\n        valores = [float(p) for p in partes]\n        if not all(math.isfinite(v) for v in valores):\n            raise ValueError(f"Label não finito: {caminho}, linha {n}")\n        id_float, xc, yc, w, h = valores\n        c = int(id_float)\n        bordas = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]\n        if id_float != c or c not in nomes or w <= 0 or h <= 0 or min(bordas) < -0.0001 or max(bordas) > 1.0001:\n            raise ValueError(f"Classe ou caixa inválida: {caminho}, linha {n}")\n        x1, y1, x2, y2 = np.clip(bordas, 0, 1)\n        objetos.append({"classe_id": c, "classe": nomes[c],\n                        "bbox": [float(x1 * largura), float(y1 * altura),\n                                 float(x2 * largura), float(y2 * altura)]})\n    return objetos\n\n\ndef carregar_home_val(dataset, nomes):\n    dataset = Path(dataset)\n    raiz_imagens = dataset / "images/val"\n    registros = []\n    for imagem in sorted(raiz_imagens.rglob("*")):\n        if not imagem.is_file() or imagem.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:\n            continue\n        relativo = imagem.relative_to(raiz_imagens)\n        if relativo.parts[0] == "sunrgbd":\n            continue\n        label = dataset / "labels/val" / relativo.with_suffix(".txt")\n        with Image.open(imagem) as im:\n            largura, altura = im.size\n            im.verify()\n        registros.append({"imagem": imagem, "nome": relativo.as_posix(),\n                          "largura": largura, "altura": altura,\n                          "gt": ler_objetos_yolo(label, largura, altura, nomes)})\n    if not registros:\n        raise ValueError("Validação Home vazia.")\n    return registros\n\n\ndef iou_matriz(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)), dtype=float)\n    g = np.asarray([o["bbox"] for o in gt], dtype=float)\n    p = np.asarray([o["bbox"] for o in pred], dtype=float)\n    inicio = np.maximum(g[:, None, :2], p[None, :, :2])\n    fim = np.minimum(g[:, None, 2:], p[None, :, 2:])\n    inter = np.prod(np.maximum(0.0, fim - inicio), axis=2)\n    ag = np.prod(np.maximum(0.0, g[:, 2:] - g[:, :2]), axis=1)\n    ap = np.prod(np.maximum(0.0, p[:, 2:] - p[:, :2]), axis=1)\n    uniao = ag[:, None] + ap[None, :] - inter\n    return np.divide(inter, uniao, out=np.zeros_like(inter), where=uniao > 0)\n\n\ndef corresponder_objetos(gt, pred):\n    matriz = iou_matriz(gt, pred)\n    candidatos = []\n    for gi, g in enumerate(gt):\n        for pi, p in enumerate(pred):\n            if g["classe_id"] == p["classe_id"] and matriz[gi, pi] >= IOU_CORRESPONDENCIA:\n                candidatos.append((float(matriz[gi, pi]), gi, pi))\n    candidatos.sort(key=lambda item: (-item[0], item[1], item[2]))\n    usados_g, usados_p, pares = set(), set(), []\n    for iou, gi, pi in candidatos:\n        if gi not in usados_g and pi not in usados_p:\n            pares.append((gi, pi, iou))\n            usados_g.add(gi)\n            usados_p.add(pi)\n    return pares, [i for i in range(len(gt)) if i not in usados_g], [i for i in range(len(pred)) if i not in usados_p]\n\n\ndef contagem_eventos(gt, pred, largura, altura):\n    """Um FN/FP pode decorrer de detecção ausente/extra ou mudança de prioridade."""\n    g = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in gt]\n    p = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in pred]\n    pares, faltantes, extras = corresponder_objetos(g, p)\n    eventos = [(g[gi], p[pi]) for gi, pi, _ in pares]\n    eventos += [(g[gi], None) for gi in faltantes]\n    eventos += [(None, p[pi]) for pi in extras]\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for ref, prev in eventos:\n        a = bool(ref and ref["prioridade"] in {"medio", "alto"})\n        b = bool(prev and prev["prioridade"] in {"medio", "alto"})\n        contagem["tp"] += int(a and b)\n        contagem["fn"] += int(a and not b)\n        contagem["fp"] += int(not a and b)\n        contagem["alta_sem_alerta"] += int(bool(ref and ref["prioridade"] == "alto" and not b))\n    return contagem\n\n\ndef avaliar_alertas(modelo, registros, nomes, device=0):\n    modelo_nomes = {int(k): str(v) for k, v in modelo.names.items()}\n    if modelo_nomes != nomes:\n        raise ValueError("Mapa de classes do modelo difere do projeto.")\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for numero, r in enumerate(registros, 1):\n        resultado = modelo.predict(str(r["imagem"]), conf=CONF_ALERTA, imgsz=IMGSZ,\n                                   iou=IOU_NMS, max_det=MAX_DETECCOES,\n                                   device=device, verbose=False, save=False)[0]\n        caixas = resultado.boxes\n        pred = [] if caixas is None else [\n            {"classe_id": int(c), "classe": nomes[int(c)], "bbox": [float(v) for v in b]}\n            for c, b in zip(caixas.cls.cpu().numpy(), caixas.xyxy.cpu().numpy())]\n        eventos = contagem_eventos(r["gt"], pred, r["largura"], r["altura"])\n        for k, v in eventos.items():\n            contagem[k] += v\n        if numero % 100 == 0:\n            print(f"Validação dos alertas: {numero}/{len(registros)}")\n    tp, fn, fp = contagem["tp"], contagem["fn"], contagem["fp"]\n    dividir = lambda n, d: n / d if d else None\n    return {"imagens": len(registros), **contagem,\n            "precision_alertas": dividir(tp, tp + fp),\n            "recall_alertas": dividir(tp, tp + fn),\n            "f1_alertas": dividir(2 * tp, 2 * tp + fp + fn),\n            "referencia": "V2 fixa nas anotações; não valida risco físico independentemente",\n            "conf": CONF_ALERTA, "iou_nms": IOU_NMS,\n            "iou_correspondencia": IOU_CORRESPONDENCIA}\n'}
CODIGO_SHA256 = {'conf14_data.py': 'f73232836278c08edf0f955ae8fe50a33cb1d4eabe2b2cc35a5f410f095bae66', 'conf14_evaluation.py': '3e64c612ffcd7cb75fdf05707affc45cafa91acd1882ab311425c2752ed01656', 'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'exp12_data.py': '0a9de7e99d302a2690f18f51924f4dc80b32861c9d2149ccfe0572ec75499f30', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'lim17_evaluation.py': 'b73248c348aee6dfe47d6aa31f3a6cafa90530a8f74898750e33d20fc80666dc', 'lim17_inputs.py': 'baba76692931493e4dfecc41e75b48bc52626b03a807ed2689a12e8b7ef4ef12', 'lim17_io.py': '6e32d0dce0cbbe0e66b076a92a0e23dc081e8e34c50e10ea0072a27649ecf466', 'lim17_protocol.py': '261094dde02a062137c5a1fcf46641cf5a0321c16e5d88facbaf4da09c182d0d', 'lim17_reports.py': '57b8b893c0dd2f2a489c741e69602dcf68fdfb302022cb89ed33b82fd6001c0e', 'lim17_run.py': '60128b4d63800dd95bc579e76916f3cbcae39acf7339ca9ce71fb02b14763035', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/politica_alertas.py': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'src/validacao_alertas.py': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d'}
SPEC_CONGELADO = {'version': '17-limiares-chair-table-1.0.0', 'model': 'candidato_10', 'tuned_class_ids': [2, 3], 'threshold_grid': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4], 'fixed_threshold_other_classes': 0.25, 'configurations': 49, 'split_seed': '17-grupos-42', 'split': 'grupos SHA256/pHash64 distancia<=4; hash do grupo, sem previsoes', 'split_fraction_target': 0.5, 'min_images_per_cohort_partition': 50, 'min_alert_objects_per_tuned_class_cohort_partition': 30, 'selection': 'somente calibracao; requisitos relativos as referencias do mesmo subconjunto', 'ranking': 'precisao Home; recall Home; menos altas omitidas Home; F1 externa; menos classes alteradas; distancia de 0.25; limiares', 'verification': 'uma configuracao congelada; sem segunda escolha se falhar', 'full_validation': 'descritiva, apos a escolha; seis limites numericos originais', 'preexposure': 'ambas as validacoes ja influenciaram o desenvolvimento', 'training': False, 'test_inference': False, 'reserved_inference': False}
PASTA_CODIGO=PASTA_LOCAL/'codigo'
PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    if hashlib.sha256(fonte.encode()).hexdigest()!=CODIGO_SHA256[nome]:
        raise RuntimeError('Módulo embutido alterado: '+nome)
    if nome in EXPECTED['codigo_12'] and CODIGO_SHA256[nome]!=EXPECTED['codigo_12'][nome]:
        raise RuntimeError('Módulo histórico difere da versão autenticada: '+nome)
    compile(fonte,nome,'exec')
    for raiz in [PASTA_CODIGO,PASTA_RESULTADOS/'codigo']:
        destino=raiz/nome;destino.parent.mkdir(parents=True,exist_ok=True)
        destino.write_text(fonte,encoding='utf-8')
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import conf14_data as dados14
import exp12_data as dados12
import lim17_protocol as protocolo17
import lim17_inputs as entradas17
import lim17_evaluation as avaliacao17
import lim17_io as arquivos17
import lim17_run as execucao17
from src import politica_alertas as politica
from src import validacao_alertas as associacao
if (core.sha256_file(politica.__file__)!=EXPECTED['politica']
        or core.sha256_file(associacao.__file__)!=EXPECTED['associador']
        or float(associacao.IOU_CORRESPONDENCIA)!=.5):
    raise RuntimeError('Política ou associador diferentes da versão congelada.')
if protocolo17.SPEC!=SPEC_CONGELADO:
    raise RuntimeError('O protocolo foi alterado; esta execução não continuará.')
def registrar_falha(exc,etapa):
    pacote=arquivos17.failure(PASTA_RESULTADOS,exc,etapa,DRIVE,REMOTO_RESULTADOS)
    print('Execução interrompida; pacote parcial com o motivo:',pacote,flush=True)
    display(FileLink(str(pacote)))
print('Código e protocolo conferidos. Confiança das outras dez classes: 0,25.')


## 3. Autenticar as entradas e a conexão

O pacote 16 identifica os caches e fornece as referências e os seis limites
numéricos históricos. O pacote 11 identifica a validação externa e a reserva.
Nenhum lote de treino ou de TEST Open Images será baixado.


In [ ]:
try:
    DRIVE=arquivos17.Drive.from_kaggle()
    if shutil.disk_usage(PASTA_LOCAL).free < 10*1024**3:
        raise RuntimeError('São necessários 10 GiB livres para conferir as entradas.')
    ENTRADAS=PASTA_LOCAL/'entradas'
    PACOTE_16=DRIVE.fetch(EXPECTED['pasta_16']+'/pacote_experimento_16.zip',
        ENTRADAS/'pacote_16.zip',EXPECTED['pacote_16'])
    FONTE16=entradas17.load_source16(PACOTE_16,EXPECTED,PASTA_RESULTADOS)
    PACOTE_11=DRIVE.fetch(EXPECTED['pasta_11']+'/pacote_diagnostico_11.zip',
        ENTRADAS/'pacote_11.zip',EXPECTED['pacote_11'])
    RESUMO11,CONFIG11,LOCAL11,MANIFESTO_NOVO=dados12.read_package11(PACOTE_11,EXPECTED['pacote_11'],
        {'antigo':EXPECTED['baseline'],'candidato':EXPECTED['candidato']},EXPECTED['dataset_novo'])
    LOCAL_ESPERADO={'pasta_drive':'datasets/validacao_independente_11_v1/'+EXPECTED['dataset_novo'],
                    'arquivo':'validacao_independente_11_695b445e15c9.zip'}
    if any(LOCAL11[k]!=v for k,v in LOCAL_ESPERADO.items()) or CONFIG11['politica_sha256']!=EXPECTED['politica']:
        raise RuntimeError('Localização ou política da validação diferente da congelada.')
    ZIP_HOME=DRIVE.fetch('datasets/homeobjects_sunrgbd_validado.zip',ENTRADAS/'home.zip',EXPECTED['home_zip'])
    ZIP_NOVO=DRIVE.fetch(LOCAL11['pasta_drive']+'/'+LOCAL11['arquivo'],ENTRADAS/'validacao_11.zip',EXPECTED['zip_novo'])
    CONGELADOS=[(p,core.sha256_file(p)) for p in [PACOTE_16,PACOTE_11,ZIP_HOME,ZIP_NOVO]]
    for nome in MODULOS_EMBUTIDOS:CONGELADOS.append((PASTA_CODIGO/nome,CODIGO_SHA256[nome]))
    core.save_json(PASTA_RESULTADOS/'entradas_conferidas.json',
        [{'arquivo':Path(p).name,'sha256':sha} for p,sha in CONGELADOS[:4]])
    print('Pacotes, ZIPs de dados, política e referências autenticados.')
except Exception as exc:
    registrar_falha(exc,'entradas e conexão');raise


## 4. Preservar os rótulos e fixar os grupos de calibração/verificação

Extrai apenas 456 Home VAL e 1.000 VAL externas. Os arquivos de treino, TEST e
os 300 itens da reserva não entram na avaliação. Imagens com mesmo SHA-256 ou
pHash a distância de até quatro bits são mantidas no mesmo grupo, inclusive
por ligação transitiva e entre as duas fontes.

O grupo recebe a partição por hash com semente fixa, sem ler previsões.
A divisão não é repetida para obter resultados favoráveis. Exige pelo menos
50 imagens e 30 objetos de alerta de cada classe ajustada em cada coorte/parte.
Essa separação adicional não apaga a exposição anterior às validações.


In [ ]:
try:
    HOME_VAL,HASHES_HOME,congelados,AUDITORIA_HOME=dados14.extract_home_val(
        ZIP_HOME,PASTA_LOCAL/'home_val',EXPECTED['home_zip'],EXPECTED['test_home'],EXPECTED['politica'])
    CONGELADOS+=congelados
    VAL_NOVA,IDS_RESERVA,congelados=dados12.load_frozen_validation(ZIP_NOVO,PASTA_LOCAL/'validacao',
        EXPECTED['zip_novo'],EXPECTED['dataset_novo'],MANIFESTO_NOVO)
    CONGELADOS+=congelados
    if len(IDS_RESERVA)!=300 or any('reserva_futura' in p.parts for p in (PASTA_LOCAL/'validacao').rglob('*')):
        raise RuntimeError('Identidades da reserva incompatíveis ou reserva extraída.')
    if {r['sha256'] for r in MANIFESTO_NOVO['arquivos']}&HASHES_HOME:
        raise RuntimeError('Coincidência de bytes entre Home e a validação externa/reserva.')
    COORTES={'home_val_historica':HOME_VAL,'openimages_val_nova':VAL_NOVA}
    entradas17.validate_source_records(COORTES,FONTE16)
    PARTICOES,DIVISAO,AUDITORIA_DIVISAO=protocolo17.group_split(COORTES,PASTA_RESULTADOS)
    SUPORTE=protocolo17.check_split_support(PARTICOES,policy=politica)
    core.save_csv(PASTA_RESULTADOS/'suporte_classes_por_particao.csv',SUPORTE)
    core.save_json(PASTA_RESULTADOS/'auditoria_home_val.json',AUDITORIA_HOME)
    core.save_json(PASTA_RESULTADOS/'protocolo_congelado_17.json',SPEC_CONGELADO)
    REGISTROS=[]
    for parte in protocolo17.PARTITIONS:
        for coorte in protocolo17.COHORTS:
            for r in PARTICOES[parte][coorte]:
                registro={k:r[k] for k in ['id','name','role','coorte','width','height','sha256',
                    'label_sha256','known_mask','gt','partition','group','phash']}
                REGISTROS.append(registro)
    execucao17.save_gz(PASTA_RESULTADOS/'registros_validacao.json.gz',REGISTROS)
    DRIVE.backup(PASTA_RESULTADOS,REMOTO_RESULTADOS)
    print('Divisão fixada antes dos novos limiares:',AUDITORIA_DIVISAO['counts'])
    print('GT e cobertura preservados; reserva e testes fechados.')
except Exception as exc:
    registrar_falha(exc,'dados e divisão agrupada');raise


## 5. Recuperar os quatro caches completos

Cada cache é conferido contra o SHA-256 registrado no pacote 16 e contra os
bytes, a ordem, as dimensões e a cobertura das imagens atuais. A reutilização
em CPU não exige ter o mesmo hardware que produziu as previsões congeladas.

Se o cache completo estiver ausente ou inválido, faz apenas inferência com o
peso autenticado e Ultralytics 8.4.150. O cache recriado é salvo no Drive.
Antes da busca, os contadores por imagem e estados dos objetos em 0,25 terão
de reproduzir o pacote 16; divergência interrompe a execução.


In [ ]:
try:
    DRIVE.restore(REMOTO_RESULTADOS+'/caches_recriados',PASTA_RESULTADOS/'caches_recriados')
    CACHES={};METADADOS_CACHES={}
    for modelo in entradas17.MODELS:
        CACHES[modelo]={};METADADOS_CACHES[modelo]={}
        for coorte,registros in COORTES.items():
            pred,meta=entradas17.obtain_cache(modelo,coorte,registros,FONTE16,EXPECTED,DRIVE,
                PASTA_LOCAL,PASTA_RESULTADOS)
            CACHES[modelo][coorte]=pred;METADADOS_CACHES[modelo][coorte]=meta
            minimo=.10 if modelo=='candidato_10' else .25
            compacto=PASTA_RESULTADOS/'previsoes_compactas'/modelo/(coorte+'.jsonl.gz')
            compacto.parent.mkdir(parents=True,exist_ok=True)
            with gzip.open(compacto,'wt',encoding='utf-8',compresslevel=4) as f:
                for r in registros:
                    f.write(json.dumps({'id':r['id'],'image_sha256':r['sha256'],'label_sha256':r['label_sha256'],
                        'confianca_minima':minimo,'predicoes':[v for v in pred[r['id']] if v['confianca']>=minimo]},ensure_ascii=False)+'\n')
            DRIVE.backup(PASTA_RESULTADOS,REMOTO_RESULTADOS)
    core.save_json(PASTA_RESULTADOS/'origem_caches.json',METADADOS_CACHES)
    print('Quatro caches prontos. A grade usa somente o candidato 10; baseline é referência fixa.')
except Exception as exc:
    registrar_falha(exc,'recuperação e autenticação dos caches');raise


## 6. Identificar a execução e recuperar cálculos compatíveis

Dados, divisão, código, protocolo, ambiente de avaliação e hashes dos caches
identificam a execução. Fronteiras de calibração completas podem ser recuperadas
somente com contexto e SHA iguais. A escolha congelada é imutável: o resultado
da verificação não poderá ser usado para selecionar outra configuração.


In [ ]:
try:
    dados12.verify_frozen(CONGELADOS)
    CONTEXTO={'protocol':SPEC_CONGELADO,'source16_sha256':EXPECTED['pacote_16'],
        'dataset_external_sha256':EXPECTED['dataset_novo'],'home_sha256':EXPECTED['home_zip'],
        'split_sha256':AUDITORIA_DIVISAO['split_sha256'],'code_sha256':CODIGO_SHA256,
        'cache_sha256':{m:{c:meta['cache_sha256'] for c,meta in values.items()} for m,values in METADADOS_CACHES.items()},
        'evaluation_environment':AMBIENTE}
    CONTEXT_KEY=protocolo17.digest(CONTEXTO)
    NOME_EXPERIMENTO='limiares17_'+CONTEXT_KEY[:24]
    PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
    PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
    shutil.copytree(PASTA_BOOT,PASTA_RESULTADOS,dirs_exist_ok=True)
    REMOTO_RESULTADOS='resultados/'+NOME_EXPERIMENTO
    DRIVE.restore(REMOTO_RESULTADOS,PASTA_RESULTADOS)
    protocolo17.immutable_json(PASTA_RESULTADOS/'contexto_execucao_17.json',{'context_key':CONTEXT_KEY,**CONTEXTO})
    for nome,sha in CODIGO_SHA256.items():
        if core.sha256_file(PASTA_RESULTADOS/'codigo'/nome)!=sha:raise RuntimeError('Código recuperado incompatível: '+nome)
    if core.sha256_file(PASTA_RESULTADOS/'regra_selecao_congelada_12.json')!=EXPECTED['regra_12_sha256']:
        raise RuntimeError('Regra histórica alterada na recuperação.')
    erro_antigo=PASTA_RESULTADOS/'execucao_interrompida.json'
    if erro_antigo.is_file():
        historico=PASTA_RESULTADOS/'historico_falhas';historico.mkdir(exist_ok=True)
        erro_antigo.replace(historico/('falha_anterior_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.json'))
    def salvar_resultados(etapa):
        print('Etapa:',etapa,flush=True)
        return DRIVE.backup(PASTA_RESULTADOS,REMOTO_RESULTADOS)
    salvar_resultados('contexto e protocolo antes da reprodução e da busca')
    print('Execução:',NOME_EXPERIMENTO)
except Exception as exc:
    registrar_falha(exc,'contexto e recuperação');raise


## 7. Reproduzir 0,25, escolher em calibração e verificar uma configuração

Primeiro reproduz baseline e candidato 10 em ambas as validações. Depois, cada
parte recebe critérios derivados das suas próprias referências em 0,25: Home
deve superar a precisão do baseline sem perder recall ou aumentar omissões altas;
a externa deve preservar as três dimensões do candidato 10.

A grade de 49 combinações só recebe imagens de calibração. A avaliação por
classe refaz correspondências em cada limiar; equivalência com a avaliação
integral em 0,25 e com a escolha selecionada é obrigatória.

Uma escolha elegível é gravada e salva no Drive **antes** da verificação. Se
falhar, não há segunda escolha. Sem elegível em calibração, nenhuma configuração
nova é testada na verificação. O conjunto completo é descrito somente após a
escolha, conservando os seis limites numéricos históricos como exigência adicional.


In [ ]:
try:
    dados12.verify_frozen(CONGELADOS)
    if protocolo17.SPEC!=SPEC_CONGELADO:raise RuntimeError('Protocolo alterado após o congelamento.')
    RESUMO17,PACOTE_FINAL=execucao17.run(COORTES,CACHES,FONTE16,PARTICOES,DIVISAO,AUDITORIA_DIVISAO,
        politica,associacao.corresponder_objetos,PASTA_RESULTADOS,CONTEXT_KEY,salvar_resultados)
    dados12.verify_frozen(CONGELADOS)
    print(RESUMO17['mensagem'])
    print('NOTEBOOK17CONCLUIDO',flush=True)
    print('Pacote de resultados:',PACOTE_FINAL,flush=True)
    display(FileLink(str(PACOTE_FINAL)))
    display(HTML((PASTA_RESULTADOS/'relatorio_calibracao_17.html').read_text(encoding='utf-8')))
except Exception as exc:
    registrar_falha(exc,'reprodução, seleção e verificação');raise
